# Layer 1 — Record-only Baseline

## What is Layer 1?

The dossier describes the whole project as a **ladder of increasingly relational representations**:

> Record-only attributes → relationship logic → graph structure → ownership enrichment → whatever else the team can justify

**Layer 1 is the first rung of this ladder: we only look at each third party's own row of information.**

- **Unit of analysis:** one vendor record (`record_id`). Each vendor is treated as its own "case" of size 1.
- **What we are allowed to use:** the vendor's **own** rows in every table (master data, due diligence, engagements, approvals, payments, services, addresses, institution links), plus reference tables (markets, service categories, employee roles, approval policy matrix, country risk indices, business events).
- **What we are *not* allowed to use:** **other vendors' rows**. Things like "how many vendors share my address" or "how many vendors does my sponsor manage" need other rows, so they belong to Layer 2.
- **The test we used:** if we add one new vendor to the tables and leave everyone else unchanged, none of the existing vendors' Layer 1 variables should change.

## Why do we need Layer 1?

From the dossier: *"Start from what a record-level view can already find, so that every later gain is measured against it."*

Layer 1 is not supposed to perform well on its own. All 33 cases in the practice set are **relational** anomalies (shared addresses, sponsor concentration, common beneficial owners, ...), so looking at single rows **can't find them completely** by design. The point of Layer 1 is to give us a **baseline**: every time we add relational information in a later layer, we measure the improvement against this.

## Notebook structure

| Cell | Content |
|---|---|
| 1 | What Layer 1 is (this cell) |
| 2 | Load the data; list the 67 variables we selected and which table/column each one comes from |
| 3 | Data cleaning and construction for each variable, producing the final feature table `X` |
| 4 | Explanation of how each variable was cleaned |
| 5 | What the target is, and what the risk score is |
| 6 | How we transform variables before modeling (log, winsorizing, imputation, scaling, ...) |
| 7 | Code for those transformations + building `y`, CV groups, and evaluation functions |
| 8–9 | **Model 1:** Linear Regression — the first model that comes to mind |
| 10–11 | **Model 2:** Logistic Regression — fixes the "output is not a probability" problem |
| 12–13 | **Model 3:** Decision Tree — allows thresholds and interactions |
| 14–15 | **Model 4:** Random Forest — averages many trees for stability |
| 16–17 | **Model 5:** Isolation Forest — unsupervised, no labels needed |
| 18 | **Model 6:** Multi-level Scorecard (`model_ms`) — a traditional points-based scoring approach; binning rules |
| 19 | Number of bins and thresholds for every variable |
| 20 | Points assigned to each bin of each variable |
| 21–22 | Weight calculation, and how the weights were obtained |
| 23 | `model_ms`: can it pick the 159 flagged vendors out of 1,002? |
| 24–25 | Comparison of all models (Model 1 → Model 6) |
| 26 | Conclusions |
| 27 | Export results |

In [1]:
# ============================================================
# Cell 2. Load data + list of selected variables (source table and column)
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 220, "display.max_rows", 200, "display.max_colwidth", 80)

# ---- Data path: student_bundle/ in the repo root ------------------------
# Works if the notebook sits in the repo root (Graph-Based-Detection/);
# if it's in a subfolder, we search upwards for student_bundle.
DATA = next(p / "student_bundle" for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "student_bundle").exists())

# ---- Analysis time t -----------------------------------------------------
# The practice labels have no dates, so we use the snapshot date. Once validation/test
# labels are released, set T to the end of that period and every variable is recomputed
# as of T (no future information).
T = pd.Timestamp("2026-06-30")

DATES = {
    "third_parties": ["onboarding_date", "created_date", "terminated_date"],
    "tp_due_diligence": ["dd_completed", "dd_refresh_due"],
    "tp_services": ["valid_from", "valid_to"],
    "tp_engagements": ["start_date", "end_date", "observed_at"],
    "tp_engagement_approvals": ["approval_date", "observed_at"],
    "tp_payments_monthly": ["month"],
    "tp_address_history": ["valid_from", "valid_to", "observed_at"],
    "tp_institution_links": ["valid_from", "valid_to"],
    "healthcare_institutions": [],
    "employee_roles": ["valid_from", "valid_to"],
    "markets": [],
    "service_categories": [],
    "approval_policy_matrix": [],
    "country_risk_context": ["as_of_date", "valid_to"],
    "business_events": ["start_date", "end_date"],
    "labelled_cases_TRAINING_SET": [],
}

def load(name):
    """Prefer parquet (keeps dtypes); fall back to csv."""
    pq, csv = DATA / f"{name}.parquet", DATA / f"{name}.csv"
    try:
        df = pd.read_parquet(pq)
    except Exception:
        df = pd.read_csv(csv)
    for c in DATES[name]:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

tables = {name: load(name) for name in DATES}
for name, df in tables.items():
    print(f"{name:32s} {df.shape[0]:>7,} rows  {df.shape[1]:>3} cols")

# ---- Variable list ---------------------------------------------------------
# (variable, source table, source columns, meaning)
# A variable can use several columns; "ref:" marks a reference table (not the vendor's own rows).
SPEC = [
    # ---------- third_parties (vendor master) ----------
    ("is_individual", "third_parties", ["party_type"], "Individual expert (vs. organization)"),
    ("expert_hcp_consultant", "third_parties", ["expert_type"], "Expert type: HCP consultant"),
    ("expert_investigator_kol", "third_parties", ["expert_type"], "Expert type: investigator / KOL"),
    ("expert_regulatory_access", "third_parties", ["expert_type"], "Expert type: regulatory / market access"),
    ("expert_other", "third_parties", ["expert_type"], "Expert type: other (3 rare types merged)"),
    ("has_lei", "third_parties", ["lei"], "Has an LEI"),
    ("has_registry_no", "third_parties", ["registry_no"], "Has a company registry number"),
    ("remit_country_mismatch", "third_parties", ["remit_to_country", "country"], "Remit-to country differs from operating country"),
    ("region_US", "third_parties", ["country", "ref:markets.region"], "Region: US (reference = Europe)"),
    ("region_JAPAN_CHINA", "third_parties", ["country", "ref:markets.region"], "Region: Japan/China"),
    ("region_INTL", "third_parties", ["country", "ref:markets.region"], "Region: other international"),
    ("tenure_years", "third_parties", ["onboarding_date"], "Years since onboarding, as of t"),
    ("terminated_by_t", "third_parties", ["terminated_date"], "Terminated by time t"),
    ("migrated_record", "third_parties", ["created_date"], "Record re-created in the 2022 ERP migration"),
    ("onboarded_in_event_window", "third_parties", ["onboarding_date", "ref:business_events.start_date", "ref:business_events.end_date"], "Onboarded during an acquisition or vaccine-launch window"),
    ("has_declared_group", "third_parties", ["declared_parent_group"], "Declared a parent group"),
    ("public_email", "third_parties", ["email_domain"], "Uses a public email provider"),
    # ---------- tp_due_diligence ----------
    ("q_public_official_interaction", "tp_due_diligence", ["q_public_official_interaction"], "Interacts with public officials on our behalf"),
    ("dd_sensitive_answer", "tp_due_diligence", ["q_gov_ownership", "q_pep_connected", "q_former_public_official"], "Self-reported state-owned / PEP-connected / former official (any)"),
    ("adverse_media", "tp_due_diligence", ["adverse_media"], "Adverse media flag"),
    ("screening_potential_match", "tp_due_diligence", ["screening_result"], "Screening potential match (cleared)"),
    ("dd_enhanced", "tp_due_diligence", ["dd_level"], "Enhanced due diligence"),
    ("dd_lead_days", "tp_due_diligence", ["dd_completed", "third_parties.onboarding_date"], "Days from DD completion to onboarding"),
    ("risk_rating_ord", "tp_due_diligence", ["risk_rating"], "Risk rating (0 low / 1 medium / 2 high)"),
    # ---------- tp_services ----------
    ("n_service_parents", "tp_services", ["category_code", "taxonomy_version", "ref:service_categories.parent"], "Number of service parent categories"),
    ("svc_flag_hcp", "tp_services", ["category_code", "taxonomy_version", "ref:service_categories.flags"], "Provides HCP-facing services"),
    ("svc_flag_gov", "tp_services", ["category_code", "taxonomy_version", "ref:service_categories.flags"], "Provides government-facing services"),
    ("svc_flag_int", "tp_services", ["category_code", "taxonomy_version", "ref:service_categories.flags"], "Provides intermediary services"),
    ("svc_flag_flow", "tp_services", ["category_code", "taxonomy_version", "ref:service_categories.flags"], "Provides high cash/goods flow services"),
    # ---------- tp_engagements ----------
    ("n_engagements", "tp_engagements", ["engagement_id"], "Number of engagements"),
    ("n_eng_markets", "tp_engagements", ["market"], "Number of engagement markets"),
    ("n_business_units", "tp_engagements", ["business_unit"], "Number of business units"),
    ("n_contract_types", "tp_engagements", ["contract_type"], "Number of contract types"),
    ("high_band_share", "tp_engagements", ["value_band"], "Share of high-value (band 3-4) engagements"),
    ("missing_sponsor_share", "tp_engagements", ["sponsor_employee_id"], "Share of engagements with no sponsor"),
    ("system_sponsor_share", "tp_engagements", ["sponsor_employee_id", "ref:employee_roles.role"], "Share of engagements with the system-default sponsor"),
    ("open_engagements", "tp_engagements", ["start_date", "end_date"], "Engagements still open at t"),
    ("eng_record_lag_days", "tp_engagements", ["start_date", "observed_at"], "Avg. days from engagement start to being recorded"),
    # ---------- tp_engagement_approvals ----------
    ("missing_approver_share", "tp_engagement_approvals", ["approver_employee_id", "approval_type"], "Share of financial approvals with no approver"),
    ("under_level_approvals", "tp_engagement_approvals", ["approver_employee_id", "approval_type", "tp_engagements.value_band", "ref:employee_roles.approver_level", "ref:approval_policy_matrix.required_approver_level"], "Approvals signed below the required level"),
    ("self_approvals", "tp_engagement_approvals", ["approver_employee_id", "tp_engagements.sponsor_employee_id"], "Approvals signed by the sponsor themselves"),
    ("missing_compliance_reviews", "tp_engagement_approvals", ["approval_type", "tp_engagements.service_category", "ref:service_categories.flags"], "Engagements missing a required compliance review"),
    ("approval_lead_days", "tp_engagement_approvals", ["approval_date", "tp_engagements.start_date"], "Avg. days from approval to engagement start"),
    # ---------- tp_payments_monthly ----------
    ("log_total_usd", "tp_payments_monthly", ["amount_usd"], "Total payments (log USD)"),
    ("active_months", "tp_payments_monthly", ["month"], "Months with payments"),
    ("activity_density", "tp_payments_monthly", ["month", "third_parties.onboarding_date"], "Months with payments / months since onboarding"),
    ("amount_cv", "tp_payments_monthly", ["month", "amount_usd"], "Coefficient of variation of monthly payments"),
    ("round_amount_share", "tp_payments_monthly", ["round_amount_share"], "Share of round-amount invoices"),
    ("manual_payment_share", "tp_payments_monthly", ["manual_payment_count", "payment_count"], "Share of manual payments"),
    ("invoices_per_payment", "tp_payments_monthly", ["invoice_count", "payment_count"], "Invoices per payment"),
    ("non_po_invoice_share", "tp_payments_monthly", ["channel"], "Share of non-PO invoices"),
    ("p_card_share", "tp_payments_monthly", ["channel"], "Share of p-card payments"),
    ("expense_reimbursement_share", "tp_payments_monthly", ["channel"], "Share of expense reimbursements"),
    ("n_expense_classes", "tp_payments_monthly", ["expense_classification"], "Number of expense classifications"),
    ("tov_hcp_share", "tp_payments_monthly", ["tov_category"], "Share of HCP/HCO transfer-of-value payments"),
    ("sensitive_expense_share", "tp_payments_monthly", ["expense_classification"], "Share of sensitive expenses (commissions/grants/sponsorships/gov. fees)"),
    # ---------- tp_address_history (own addresses only) ----------
    ("n_addresses", "tp_address_history", ["raw_address"], "Number of address records"),
    ("risky_address_type", "tp_address_history", ["address_type"], "Used a registered-agent / virtual-office / PO-box / coworking address"),
    ("org_at_residential", "tp_address_history", ["address_type", "third_parties.party_type"], "Organization registered at a residential address"),
    ("address_type_missing", "tp_address_history", ["address_type"], "Address type not classified"),
    # ---------- tp_institution_links ----------
    ("n_institution_links", "tp_institution_links", ["institution_id"], "Number of institution links"),
    ("public_institution_links", "tp_institution_links", ["institution_id", "ref:healthcare_institutions.public_sector"], "Links to public-sector institutions"),
    ("link_serves", "tp_institution_links", ["link_type"], "Institutions served (distributor)"),
    ("link_site_of", "tp_institution_links", ["link_type"], "Institutions as trial site"),
    ("link_affiliated_with", "tp_institution_links", ["link_type"], "Institutions the expert is affiliated with"),
    # ---------- country_risk_context (edition in force at t) ----------
    ("country_cpi_score", "country_risk_context", ["third_parties.country", "iso2", "index", "as_of_date", "value"], "Country CPI score"),
    ("country_bo_access_low", "country_risk_context", ["third_parties.country", "iso2", "index", "as_of_date", "tier"], "Country has low beneficial-ownership register access"),
]

def col_ref(table, col):
    """'col' -> 'col (#k)', where k is the 1-based column position in its table."""
    if col.startswith("ref:"):
        t, c = col[4:].split(".")
        return f"{t}.{c} (#{tables[t].columns.get_loc(c) + 1})"
    if "." in col:
        t, c = col.split(".")
        return f"{t}.{c} (#{tables[t].columns.get_loc(c) + 1})"
    return f"{col} (#{tables[table].columns.get_loc(col) + 1})"

spec = pd.DataFrame([
    {"variable": v, "source_table": t,
     "source_columns (#column position)": ", ".join(col_ref(t, c) for c in cols),
     "meaning": m}
    for v, t, cols, m in SPEC
])
spec.index = np.arange(1, len(spec) + 1)
VARIABLES = spec["variable"].tolist()

print(f"\nLayer 1 variables: {len(VARIABLES)}")
print(spec.groupby("source_table", sort=False).size().rename("n_variables").to_string())
spec

third_parties                      1,002 rows   16 cols
tp_due_diligence                   1,002 rows   11 cols
tp_services                        1,791 rows    5 cols
tp_engagements                     2,638 rows   12 cols
tp_engagement_approvals            4,298 rows    6 cols
tp_payments_monthly               34,467 rows   18 cols
tp_address_history                 1,091 rows    8 cols
tp_institution_links                 403 rows    6 cols
healthcare_institutions              213 rows    5 cols
employee_roles                     1,502 rows    9 cols
markets                               74 rows    4 cols
service_categories                    43 rows    5 cols
approval_policy_matrix                 5 rows    5 cols
country_risk_context               1,120 rows    8 cols
business_events                        8 rows    5 cols
labelled_cases_TRAINING_SET           33 rows    6 cols

Layer 1 variables: 67
source_table
third_parties              17
tp_due_diligence            7
tp_servi

,variable,source_table,source_columns (#column position),meaning
1,is_individual,third_parties,party_type (#2),Individual expert (vs. organization)
2,expert_hcp_consultant,third_parties,expert_type (#16),Expert type: HCP consultant
3,expert_investigator_kol,third_parties,expert_type (#16),Expert type: investigator / KOL
4,expert_regulatory_access,third_parties,expert_type (#16),Expert type: regulatory / market access
5,expert_other,third_parties,expert_type (#16),Expert type: other (3 rare types merged)
6,has_lei,third_parties,lei (#5),Has an LEI
7,has_registry_no,third_parties,registry_no (#6),Has a company registry number
8,remit_country_mismatch,third_parties,"remit_to_country (#15), country (#7)",Remit-to country differs from operating country
9,region_US,third_parties,"country (#7), markets.region (#3)",Region: US (reference = Europe)
10,region_JAPAN_CHINA,third_parties,"country (#7), markets.region (#3)",Region: Japan/China


In [2]:
# ============================================================
# Cell 3. Clean + build each variable, producing the final feature table X
#   The tags [C1] [C2] ... match the explanations in Cell 4
# ============================================================
tp  = tables["third_parties"].copy()
dd  = tables["tp_due_diligence"].copy()
sv  = tables["tp_services"].copy()
en  = tables["tp_engagements"].copy()
ap  = tables["tp_engagement_approvals"].copy()
pm  = tables["tp_payments_monthly"].copy()
ad  = tables["tp_address_history"].copy()
il  = tables["tp_institution_links"].copy()
hi  = tables["healthcare_institutions"]
er  = tables["employee_roles"]
mk  = tables["markets"]
sc  = tables["service_categories"]
apm = tables["approval_policy_matrix"]
cr  = tables["country_risk_context"].copy()
ev  = tables["business_events"]

# ---------------------------------------------------------------
# [C0] Time cutoff: keep only what happened before t AND was recorded before t
# ---------------------------------------------------------------
tp = tp[tp["onboarding_date"] <= T]
en = en[(en["start_date"] <= T) & (en["observed_at"] <= T)]
ap = ap[(ap["observed_at"] <= T) & ap["engagement_id"].isin(en["engagement_id"])]
pm = pm[(pm["month"] <= T) & pm["engagement_id"].isin(en["engagement_id"])]
sv = sv[sv["valid_from"] <= T]
ad = ad[(ad["valid_from"] <= T) & (ad["observed_at"] <= T)]
il = il[il["valid_from"] <= T]
cr = cr[cr["as_of_date"] <= T]

# [C0b] Make boolean columns real True/False (csv may give strings)
def to_bool(s):
    return s.map({True: True, False: False, "True": True, "False": False,
                  "true": True, "false": False, 1: True, 0: False}).fillna(False).astype(bool)
for c in ["q_gov_ownership", "q_pep_connected", "q_public_official_interaction",
          "q_former_public_official", "adverse_media"]:
    dd[c] = to_bool(dd[c])
hi = hi.assign(public_sector=to_bool(hi["public_sector"]))

idx = pd.Index(tp["record_id"], name="record_id")
t_ = tp.set_index("record_id")
X = pd.DataFrame(index=idx)

def share(num, den):
    return num / den.replace(0, np.nan)

# ===============================================================
# third_parties
# ===============================================================
X["is_individual"] = (t_["party_type"] == "individual").astype(int)

# [C1] expert_type: empty for organizations = "not applicable"; merge 3 rare types into expert_other
for e in ["hcp_consultant", "investigator_kol", "regulatory_access"]:
    X[f"expert_{e}"] = (t_["expert_type"] == e).astype(int)
X["expert_other"] = t_["expert_type"].isin(
    ["other_consultant", "patient_expert", "writer_translator"]).astype(int)

# [C2] lei / registry_no: missing = not recorded, so only use a "has one" flag
X["has_lei"] = t_["lei"].notna().astype(int)
X["has_registry_no"] = t_["registry_no"].notna().astype(int)

# [C3] Uppercase and strip country codes before comparing
country = t_["country"].str.strip().str.upper()
remit = t_["remit_to_country"].str.strip().str.upper()
X["remit_country_mismatch"] = (remit != country).astype(int)

# [C4] country -> region (67 countries is too many; 4 regions, Europe = reference group)
region = country.map(mk.set_index("market_id")["region"])
for r in ["US", "JAPAN_CHINA", "INTL"]:
    X[f"region_{r}"] = (region == r).astype(int)

# [C5] Time variables: always use onboarding_date; don't use status, recompute as of t
X["tenure_years"] = (T - t_["onboarding_date"]).dt.days / 365.25
X["terminated_by_t"] = (t_["terminated_date"] <= T).astype(int)
X["migrated_record"] = (t_["created_date"] == pd.Timestamp("2022-05-01")).astype(int)

# [C6] business_events: onboarding during acquisitions / vaccine launch -> likely benign bursts
burst_ev = ev[ev["event_name"].str.contains("acquisition|launch", case=False)]
in_ev = pd.Series(False, index=idx)
for _, e in burst_ev.iterrows():
    end = e["end_date"] if pd.notna(e["end_date"]) else e["start_date"]
    in_ev |= t_["onboarding_date"].between(e["start_date"], end)
X["onboarded_in_event_window"] = in_ev.astype(int)

X["has_declared_group"] = t_["declared_parent_group"].notna().astype(int)

# [C7] email_domain: lowercase and compare with a list of public providers
PUBLIC_EMAIL = {"gmail.com", "outlook.com", "yahoo.com", "hotmail.com", "icloud.com",
                "qq.com", "163.com", "aol.com", "gmx.de", "web.de", "yandex.ru", "mail.ru"}
X["public_email"] = t_["email_domain"].str.strip().str.lower().isin(PUBLIC_EMAIL).astype(int)

# ===============================================================
# tp_due_diligence
# ===============================================================
d = dd.set_index("record_id").reindex(idx)
X["q_public_official_interaction"] = d["q_public_official_interaction"].astype(int)
# [C8] Merge three very rare questionnaire answers into one flag
X["dd_sensitive_answer"] = (d["q_gov_ownership"] | d["q_pep_connected"] |
                            d["q_former_public_official"]).astype(int)
X["adverse_media"] = d["adverse_media"].astype(int)
X["screening_potential_match"] = (d["screening_result"] != "clear").astype(int)
X["dd_enhanced"] = (d["dd_level"] == "enhanced").astype(int)
X["dd_lead_days"] = (t_["onboarding_date"] - d["dd_completed"]).dt.days
# [C9] Ordinal categories -> numbers
X["risk_rating_ord"] = d["risk_rating"].map({"low": 0, "medium": 1, "high": 2})

# ===============================================================
# [C10] Unify service taxonomy: map both v1 and v2 codes, via service_categories,
#       to the same parent categories and risk flags
# ===============================================================
f_v2 = sc.set_index("category_code_v2")["flags"].fillna("")
f_v1 = sc.groupby("category_code_v1")["flags"].apply(lambda s: ";".join(s.fillna("")))
p_v2 = sc.set_index("category_code_v2")["parent"]
p_v1 = sc.groupby("category_code_v1")["parent"].first()

def cat_flags(code, ver):
    return f_v2.get(code, "") if ver == "v2" else f_v1.get(code, "")

def cat_parent(code, ver):
    return p_v2.get(code) if ver == "v2" else p_v1.get(code)

sv["flg"] = [cat_flags(c, v) for c, v in zip(sv["category_code"], sv["taxonomy_version"])]
sv["parent"] = [cat_parent(c, v) for c, v in zip(sv["category_code"], sv["taxonomy_version"])]
g = sv.groupby("record_id")
X["n_service_parents"] = g["parent"].nunique()
for f in ["hcp", "gov", "int", "flow"]:
    X[f"svc_flag_{f}"] = g["flg"].apply(lambda s: s.str.contains(f).any()).astype(float)

# ===============================================================
# tp_engagements
# ===============================================================
en["flg"] = [cat_flags(c, v) for c, v in zip(en["service_category"], en["taxonomy_version"])]
# [C11] Identify the system-default sponsor (assigned in the ERP migration), not a real sponsor
sys_sponsor = set(er.loc[er["role"] == "system_default", "employee_id"])
en["sys_sponsor"] = en["sponsor_employee_id"].isin(sys_sponsor).astype(int)
en["no_sponsor"] = en["sponsor_employee_id"].isna().astype(int)
# [C12] Empty end_date = still open at the snapshot (not missing)
en["open_at_t"] = (en["end_date"].isna() | (en["end_date"] > T)).astype(int)
en["obs_lag"] = (en["observed_at"] - en["start_date"]).dt.days
en["high_band"] = (en["value_band"] >= 3).astype(int)

g = en.groupby("record_id")
X["n_engagements"] = g.size()
X["n_eng_markets"] = g["market"].nunique()
X["n_business_units"] = g["business_unit"].nunique()
X["n_contract_types"] = g["contract_type"].nunique()
X["high_band_share"] = g["high_band"].mean()
X["missing_sponsor_share"] = g["no_sponsor"].mean()
X["system_sponsor_share"] = g["sys_sponsor"].mean()
X["open_engagements"] = g["open_at_t"].sum()
X["eng_record_lag_days"] = g["obs_lag"].mean()

# ===============================================================
# tp_engagement_approvals (+ employee_roles, approval_policy_matrix)
# ===============================================================
lvl = er.set_index("employee_id")["approver_level"]
req = apm.set_index("value_band")["required_approver_level"]
a = ap.merge(en[["engagement_id", "record_id", "value_band", "sponsor_employee_id",
                 "start_date"]], on="engagement_id")
fin = a[a["approval_type"] == "financial"].copy()
fin["no_approver"] = fin["approver_employee_id"].isna().astype(int)
# [C13] No approver -> level unknown -> not counted as "under level" (NaN < x is False)
fin["under_level"] = (fin["approver_employee_id"].map(lvl) <
                      fin["value_band"].map(req)).astype(int)
a["self_approved"] = (a["approver_employee_id"] == a["sponsor_employee_id"]).astype(int)
a["lead_days"] = (a["start_date"] - a["approval_date"]).dt.days
needs_cr = en[en["flg"].str.contains("hcp|gov|int")]
has_cr = set(a.loc[a["approval_type"] == "compliance_review", "engagement_id"])
needs_cr = needs_cr.assign(missing_cr=(~needs_cr["engagement_id"].isin(has_cr)).astype(int))

X["missing_approver_share"] = fin.groupby("record_id")["no_approver"].mean()
X["under_level_approvals"] = fin.groupby("record_id")["under_level"].sum()
X["self_approvals"] = a.groupby("record_id")["self_approved"].sum()
X["missing_compliance_reviews"] = needs_cr.groupby("record_id")["missing_cr"].sum()
X["approval_lead_days"] = a.groupby("record_id")["lead_days"].mean()

# ===============================================================
# tp_payments_monthly
# ===============================================================
pm["tov_hcp"] = (pm["tov_category"] != "not_applicable").astype(int)
pm["sensitive_exp"] = pm["expense_classification"].isin(
    ["commissions", "grants_donations", "sponsorships", "government_fees"]).astype(int)
for ch in ["non_po_invoice", "p_card", "expense_reimbursement"]:
    pm[f"ch_{ch}"] = (pm["channel"] == ch).astype(int)

g = pm.groupby("record_id")
# [C14] Amounts are heavily right-skewed -> log1p
X["log_total_usd"] = np.log1p(g["amount_usd"].sum())
X["active_months"] = g["month"].nunique()
X["activity_density"] = share(X["active_months"], X["tenure_years"] * 12).clip(upper=1)
# [C15] Aggregate by month first, then compute variation; one month only -> std undefined -> missing
monthly = pm.groupby(["record_id", "month"])["amount_usd"].sum().groupby("record_id")
X["amount_cv"] = monthly.std() / monthly.mean()
X["round_amount_share"] = g["round_amount_share"].mean()
X["manual_payment_share"] = share(g["manual_payment_count"].sum(), g["payment_count"].sum())
X["invoices_per_payment"] = share(g["invoice_count"].sum(), g["payment_count"].sum())
for ch in ["non_po_invoice", "p_card", "expense_reimbursement"]:
    X[f"{ch}_share"] = g[f"ch_{ch}"].mean()
X["n_expense_classes"] = g["expense_classification"].nunique()
X["tov_hcp_share"] = g["tov_hcp"].mean()
X["sensitive_expense_share"] = g["sensitive_exp"].mean()

# ===============================================================
# tp_address_history (own address types only, no comparison with other vendors)
# ===============================================================
# [C16] Lowercase address_type; flag missing separately instead of treating it as "safe"
ad["address_type"] = ad["address_type"].str.strip().str.lower()
ad["risky_type"] = ad["address_type"].isin(
    ["registered_agent", "virtual_office", "po_box", "coworking"]).astype(int)
ad["residential"] = (ad["address_type"] == "residential").astype(int)
ad["type_missing"] = ad["address_type"].isna().astype(int)
g = ad.groupby("record_id")
X["n_addresses"] = g.size()
X["risky_address_type"] = g["risky_type"].max()
X["org_at_residential"] = g["residential"].max() * (1 - X["is_individual"])
X["address_type_missing"] = g["type_missing"].max()

# ===============================================================
# tp_institution_links (+ healthcare_institutions)
# ===============================================================
il = il.merge(hi[["institution_id", "public_sector"]], on="institution_id", how="left")
g = il.groupby("record_id")
X["n_institution_links"] = g.size()
X["public_institution_links"] = g["public_sector"].sum()
for lt in ["SERVES", "SITE_OF", "AFFILIATED_WITH"]:
    X[f"link_{lt.lower()}"] = il[il["link_type"] == lt].groupby("record_id").size()

# ===============================================================
# [C17] country_risk_context: latest edition in force at t
# ===============================================================
def latest(index_name, col):
    s = cr[cr["index"] == index_name].sort_values("as_of_date").groupby("iso2")[col].last()
    return country.map(s)

X["country_cpi_score"] = latest("CPI", "value")
X["country_bo_access_low"] = (latest("BO_REGISTER_ACCESS", "tier") == "low").astype(int)

# ===============================================================
# [C18] Missing values
#   (a) counts: no rows = 0 (truly none)
#   (b) shares/averages: no denominator = undefined -> keep NaN, impute median at model time
# ===============================================================
COUNT_VARS = ["n_service_parents", "n_engagements", "n_eng_markets", "n_business_units",
              "n_contract_types", "open_engagements", "under_level_approvals",
              "self_approvals", "missing_compliance_reviews", "active_months",
              "n_expense_classes", "n_addresses", "n_institution_links",
              "public_institution_links", "link_serves", "link_site_of",
              "link_affiliated_with", "svc_flag_hcp", "svc_flag_gov", "svc_flag_int",
              "svc_flag_flow", "risky_address_type", "org_at_residential"]
X[COUNT_VARS] = X[COUNT_VARS].fillna(0)

X = X[VARIABLES].astype(float)          # same column order as the list in Cell 2

# ---- Checks ----------------------------------------------------------
assert list(X.columns) == VARIABLES and X.index.is_unique
check = pd.DataFrame({
    "missing_share": X.isna().mean().round(3),
    "n_unique": X.nunique(),
    "min": X.min().round(2),
    "median": X.median().round(2),
    "max": X.max().round(2),
})
print(f"X: {X.shape[0]} records x {X.shape[1]} variables (as of {T.date()})")
print("variables with missing values (filled with median at model time):")
print(check[check["missing_share"] > 0].to_string())
check

X: 1002 records x 67 variables (as of 2026-06-30)
variables with missing values (filled with median at model time):
                             missing_share  n_unique   min  median    max
high_band_share                      0.015        22  0.00    0.00   1.00
missing_sponsor_share                0.015        17  0.00    0.00   1.00
system_sponsor_share                 0.015         9  0.00    0.00   1.00
eng_record_lag_days                  0.015       106  0.00    2.50  22.00
missing_approver_share               0.015        19  0.00    0.00   1.00
approval_lead_days                   0.015       258  3.00   17.17  33.50
log_total_usd                        0.017       985  5.22   12.76  20.42
activity_density                     0.018       863  0.01    0.35   1.00
amount_cv                            0.029       973  0.01    0.52   2.26
round_amount_share                   0.017       940  0.01    0.08   0.23
manual_payment_share                 0.017       294  0.00    0.01   0

,missing_share,n_unique,min,median,max
is_individual,0.000,2,0.00,0.00,1.00
expert_hcp_consultant,0.000,2,0.00,0.00,1.00
expert_investigator_kol,0.000,2,0.00,0.00,1.00
expert_regulatory_access,0.000,2,0.00,0.00,1.00
expert_other,0.000,2,0.00,0.00,1.00
has_lei,0.000,2,0.00,0.00,1.00
has_registry_no,0.000,2,0.00,0.00,1.00
remit_country_mismatch,0.000,2,0.00,0.00,1.00
region_US,0.000,2,0.00,0.00,1.00
region_JAPAN_CHINA,0.000,2,0.00,0.00,1.00


## Cell 4 · Data Cleaning Notes

The labels below match the `[C*]` tags in the Cell 3 code. We only list variables that needed cleaning or special handling; the rest are direct values or simple counts.

| Tag | Variables | Problem we found | What we did |
|---|---|---|---|
| **C0** | All variables | The data has two clocks: *valid time* (when something was actually true) and *observed time* (when the company recorded it). Filtering on only one of them can leak future information | We only keep rows that **happened before t *and* were recorded before t** |
| **C0b** | Questionnaire answers, adverse_media, public_sector | Booleans may be read in as strings like `"True"` from csv | Convert everything to True/False |
| **C1** | expert_* | ① `expert_type` is empty for organizations, which means "not applicable", not missing; ② patient_expert (8) and writer_translator (5) are too rare | ① Organizations get 0 on every expert_* column; ② we merged other_consultant, patient_expert, and writer_translator into `expert_other` (24 people) |
| **C2** | has_lei, has_registry_no | LEI is missing for 67% and registry number for 64%; missing means "not recorded", not "doesn't exist" | We don't use the raw IDs, only a 0/1 "has one" flag |
| **C3** | remit_country_mismatch | Country codes might differ in case or whitespace | Uppercase and strip before comparing |
| **C4** | region_* | 67 countries, many with only a few vendors, so one-hot encoding would be very sparse | Map countries to 4 regions using `markets.csv`; Europe is the reference group (no column), to avoid perfect collinearity with the intercept |
| **C5** | tenure_years, terminated_by_t, migrated_record | ① 169 records have `created_date` = 2022-05-01 because of the ERP migration, so it doesn't reflect when the relationship started; ② `status` is the status on the snapshot date (2026-06-30), so using it means knowing the future | ① Tenure always uses `onboarding_date`; migrated records get their own 0/1 flag; ② we don't use `status` and instead compute "`terminated_date` ≤ t" |
| **C6** | onboarded_in_event_window | Acquisitions and the vaccine launch caused batches of vendors to be onboarded together, which is benign | Flag onboarding dates that fall in those windows from `business_events.csv` |
| **C7** | public_email | 268 records share an email domain, mostly public providers like gmail or outlook | Lowercase and compare against a list of public providers; the raw domain itself isn't a variable (it's for linking vendors in Layer 2) |
| **C8** | dd_sensitive_answer | q_gov_ownership (4 True), q_pep_connected (5), and q_former_public_official (3) are each too rare on their own | Combined into "any of the three is True" |
| **C9** | risk_rating_ord | Text categories low / medium / high | Ordinal mapping to 0 / 1 / 2; `dd_level` overlaps almost completely with it, so we only kept an "enhanced DD" 0/1 flag |
| **C10** | n_service_parents, svc_flag_* | Service categories were re-coded on 2024-01-01, so v1 and v2 codes are mixed; v1 → v2 is one-to-many | Map both code sets to the same parent categories and risk flags using `service_categories.csv`; when one v1 code maps to several v2 codes, we take the union of their flags |
| **C11** | system_sponsor_share | During the ERP migration, engagements without a sponsor were assigned to a "system default" account, which looks like a real sponsor | Identify it via `role = system_default` in `employee_roles` and compute its share separately |
| **C12** | open_engagements | `end_date` is empty for 26% of engagements | Empty means "still open at the snapshot" (per the data dictionary), not missing |
| **C13** | under_level_approvals | The approver is missing for 13.7% of approvals, so we can't know their level | Approvals with no approver are not counted as "under level"; the missingness itself is captured by `missing_approver_share` |
| **C14** | log_total_usd | Total payments are heavily right-skewed (a few vendors have huge amounts) | Take log(1 + x) |
| **C15** | amount_cv | The payments table is at the vendor × engagement × month × expense type × channel level, so one month can have several rows | Aggregate by month first, then compute the coefficient of variation; vendors with only one month of payments can't have one, so it stays missing |
| **C16** | risky_address_type, org_at_residential, address_type_missing | ① Case might be inconsistent; ② `address_type` is missing for 38% of rows | ① Lowercase; ② we **don't treat missing as "safe"** and add a separate `address_type_missing` flag |
| **C17** | country_cpi_score, country_bo_access_low | CPI has a new edition every year, and we shouldn't use the latest edition to analyze the past | Use the latest edition with `as_of_date` ≤ t |
| **C18** | All aggregated variables | If a vendor has no rows in a table, the aggregate is empty | **Counts**: no rows = 0 (truly none); **shares/averages**: no denominator = undefined, so we leave them missing and impute the median at modeling time |

### Columns we excluded on purpose

| Column | Why |
|---|---|
| record_id, legal_name, tax_id, phone, raw email_domain, raw_address, all employee IDs | IDs and contact details have no meaning on their own; their job is to **link vendors together**, which is Layer 2 |
| status | Snapshot status, leaks future information (replaced by terminated_by_t) |
| dd_level | Almost identical to risk_rating |
| dd_refresh_due | Only the first DD round is recorded, so we can't tell whether a refresh was actually overdue |
| address_role | Always "registered", no information |
| max_value_band, max_invoice_band, cross_border_*, n_paying_entities, burstiness | Highly correlated with variables we kept (\|ρ\| ≥ 0.9) |
| FMV cap exceedances, invoice above contract band, sanctioned country | Fewer than 10 vendors each, too rare |

## Cell 5 · Target and Risk Score

### 1. Unit of analysis: one vendor = one case

Layer 1 only looks at single rows and has no relational information, so **we can't group vendors together** (grouping needs relationships like shared addresses or shared sponsors, which is Layer 2). So in Layer 1, **each vendor is its own case of size 1**.

This is also the simplest version of the general framework we want to use later: every layer will "generate candidate cases → score each case", with the candidate cases getting bigger and the case features getting richer as we go up.

### 2. Target $y$

$$
y_i = \begin{cases} 1 & \text{vendor } i \text{ is a member of any case in labelled\_cases\_TRAINING\_SET} \\ 0 & \text{otherwise} \end{cases}
$$

- 33 labelled cases expand to 159 vendors with $y = 1$, which is 15.9% of the data.
- When each case has size 1, "does this case match a labelled case" is the same as "does this vendor belong to a labelled case", so the case-level target reduces to the formula above.

### 3. The risk score is the model **output**, not the target

$$
\text{risk}_i = f(x_i)
$$

- **Supervised models** (Models 1–4: linear regression, logistic regression, decision tree, random forest): trained on $y$, and $\text{risk}_i$ is the predicted likelihood that $y_i = 1$.
- **Unsupervised model** (Model 5: isolation forest): $\text{risk}_i$ is how anomalous the vendor looks. It **never sees $y$** during training; we only use $y$ to evaluate it. We need an unsupervised model because the main dataset (~20,000 records) **has no labels**, validation labels only come mid-project, and the test period is scored blind by GSK.
- As the dossier points out, unsupervised scores are **rankings**, not probabilities, unless they are calibrated.

### 4. Why we don't use `risk_rating` as the target

`risk_rating` is the score the company assigned at onboarding using its **existing rules**. If we used it as the target, the model would just learn to imitate those rules, but the whole point of the project is to find things the existing rules miss. It's also already one of our input variables, so it can't be both an input and the answer.

### 5. Two things we need to control for

- **Members of the same case must stay together.** For cross-validation we use **StratifiedGroupKFold**, so all members of a case land in the same fold. Otherwise the model would have "seen part of the answer". Two vendors belong to two cases each, so those cases are merged into one group.
- **Shortcut variables.** Some variables separate the practice set really well, but this is probably a property of the practice set rather than a real pattern: region and CPI (the practice cases are concentrated in international markets), termination status and tenure (the practice cases are concentrated in recent years), and risk rating (the company's own judgment). So we run two feature sets:
  - **Full**: all 67 variables
  - **No-shortcut**: drops those 7 variables, leaving 60

### 6. Evaluation metrics

An analyst only reviews the top k vendors on the list, and the classes are imbalanced, so following the dossier we evaluate the **quality of the ranking**, not accuracy:

| Metric | Meaning |
|---|---|
| Precision@k | Of the top k vendors, how many are true positives |
| Recall@k | Of all positives, how many made it into the top k |
| Average Precision (AP) | Summarizes precision along the whole ranking |
| Case hit@k | Of the 33 labelled cases, how many have **at least one member** in the top k |
| Member recall@k | On average, what fraction of each case's members are in the top k |

## Cell 6 · Preparing Variables for Modeling (code in Cell 7)

The cleaning in Cell 3 dealt with **problems in the data itself** (formats, versions, what missing means, time leakage). This step deals with **the shape of the variables before they go into a model**. Different models need different things: linear models (linear and logistic regression) are sensitive to scale and extreme values, while tree models (decision tree, random forest) only care about the **order** of values, so they barely need any of this.

### 6.1 Transformations

| # | Transformation | Variables | Why | Used for |
|---|---|---|---|---|
| T1 | **log(1 + x)** | Total payments (already logged in Cell 3), plus 14 right-skewed count/day variables: `n_engagements`, `open_engagements`, `active_months`, `n_eng_markets`, `n_addresses`, `n_institution_links`, `link_serves`, `link_site_of`, `public_institution_links`, `under_level_approvals`, `self_approvals`, `missing_compliance_reviews`, `dd_lead_days`, `eng_record_lag_days` | All of these have skewness above 2. For example, `link_serves` has skewness 15.3: 99% of vendors have ≤ 6, but the max is 47. Without the log, one or two extreme values would dominate the linear model's coefficients. We use log(1+x) because there are zeros | Linear & logistic regression |
| T2 | **Winsorize at the 1st and 99th percentiles** | All continuous variables | Further limits extreme values, e.g. `dd_lead_days` has a 99th percentile of 309 days but a max of 778. **The cutoffs are computed on the training fold only** and then applied to the test fold, so no test information leaks in | Linear & logistic regression |
| T3 | **Median imputation + missing indicators** | 17 share/average variables (1.5%–2.9% missing) | The missing values all come from vendors with no engagements or no payments (about 15–29 vendors), where the share is undefined. Along with median imputation we add a 0/1 "was missing" column, so the model can decide whether "no payment history" carries information. The median is also computed on the training fold only | All models |
| T4 | **Standardization (z-score)**: $z = (x - \bar x)/s$ | All variables | Makes coefficients comparable: each coefficient means "how much the score changes when this variable goes up by one standard deviation" | Linear & logistic regression, isolation forest |
| T5 | **Class weighting**: `class_weight="balanced"` | Target $y$ | Only 15.9% of vendors are positive. Without weighting, the model leans towards predicting 0 for everyone. With weighting, each positive counts about 5.3 times as much as a negative (= 843 / 159) | Logistic regression, decision tree, random forest |
| T6 | **Two feature sets** | Full (67) / No-shortcut (60) | Remove the 7 possible shortcut variables (3 region dummies, CPI, termination status, tenure, risk rating) to check whether the model relies on them | All models |

**Why trees don't get T1, T2, T4:** every split in a tree is "variable ≤ some value". Log transforms, winsorizing, and standardizing don't change the order of the values, so the tree would be exactly the same and there's no point doing them.

### 6.2 Cross-validation

- **StratifiedGroupKFold with 5 folds:** all members of a labelled case go into the same fold (see Cell 5), and each fold has roughly the same share of positives.
- Each vendor's score comes from a model that **never saw that vendor** (out-of-fold predictions).
- We repeat everything with **5 random seeds** and report the mean and standard deviation, to make sure the results are stable and not just one lucky split.
- Ties in the ranking are broken randomly (the decision tree only produces a handful of distinct scores, so there are a lot of ties).

In [3]:
# ============================================================
# Cell 7. y, CV groups, variable transformations, CV runner and evaluation
# ============================================================
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score

# ---------- target y ----------
cases = tables["labelled_cases_TRAINING_SET"].copy()
cases["members"] = cases["member_record_ids"].str.split("|")
membership = cases[["case_id", "members"]].explode("members").rename(columns={"members": "record_id"})
membership = membership[membership["record_id"].isin(X.index)]
y = pd.Series(X.index.isin(membership["record_id"]).astype(int), index=X.index, name="y")

# ---------- groups: same case -> same group; cases sharing members are merged ----------
parent = {}
def find(a):
    parent.setdefault(a, a)
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a
for _, grp in membership.groupby("record_id")["case_id"]:
    ids = grp.tolist()
    for other in ids[1:]:
        parent[find(ids[0])] = find(other)
case_group = (membership.assign(group=membership["case_id"].map(find))
              .drop_duplicates("record_id").set_index("record_id")["group"])
rid = pd.Series(X.index, index=X.index)
groups = rid.map(case_group).fillna(rid)
print(f"records {len(y)}, positives {y.sum()} ({y.mean():.1%}), "
      f"CV groups among positives {groups[y == 1].nunique()}")

# ---------- T1: log(1+x) ----------
LOG_VARS = ["n_engagements", "open_engagements", "active_months", "n_eng_markets",
            "n_addresses", "n_institution_links", "link_serves", "link_site_of",
            "public_institution_links", "under_level_approvals", "self_approvals",
            "missing_compliance_reviews", "dd_lead_days", "eng_record_lag_days"]
X_lin = X.copy()
X_lin[LOG_VARS] = np.log1p(X_lin[LOG_VARS])
skew = pd.DataFrame({"skew_before": X[LOG_VARS].skew(), "skew_after_log": X_lin[LOG_VARS].skew()}).round(2)
print("\nT1 log(1+x):"); print(skew.to_string())

# ---------- T2: winsorize (quantiles from the training fold only) ----------
class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper
    def fit(self, Z, y=None):
        self.lo_ = np.nanquantile(Z, self.lower, axis=0)
        self.hi_ = np.nanquantile(Z, self.upper, axis=0)
        return self
    def transform(self, Z):
        return np.clip(Z, self.lo_, self.hi_)

# Linear models: T3 impute + missing indicators -> T2 winsorize -> T4 standardize
def linear_prep():
    return [SimpleImputer(strategy="median", add_indicator=True), Winsorizer(), StandardScaler()]
# Tree models: T3 only
def tree_prep():
    return [SimpleImputer(strategy="median", add_indicator=True)]

# ---------- T6: two feature sets ----------
SHORTCUT_VARS = ["region_US", "region_JAPAN_CHINA", "region_INTL", "country_cpi_score",
                 "terminated_by_t", "tenure_years", "risk_rating_ord"]
FEATURE_SETS = {"Full": VARIABLES,
                "No-shortcut": [v for v in VARIABLES if v not in SHORTCUT_VARS]}
SEEDS = [0, 1, 2, 3, 4]

def oof_scores(model, Xs, seed):
    """Out-of-fold scores from 5-fold grouped CV (supervised models)."""
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    score = pd.Series(np.nan, index=Xs.index)
    for tr, te in cv.split(Xs, y, groups):
        m = clone(model).fit(Xs.iloc[tr], y.iloc[tr])
        score.iloc[te] = (m.predict_proba(Xs.iloc[te])[:, 1] if hasattr(m, "predict_proba")
                          else m.predict(Xs.iloc[te]))
    return score

# ---------- Evaluation ----------
KS = [10, 25, 50, 100]
case_members = cases.set_index("case_id")["members"].apply(lambda m: [r for r in m if r in X.index])

def evaluate(score, seed=0):
    tie = np.random.default_rng(seed).random(len(score))          # break ties randomly
    order = score.index[np.lexsort((tie, -score.to_numpy()))]
    yy = y.loc[order].to_numpy()
    out = {"AP": average_precision_score(y, score.loc[y.index])}
    for k in KS:
        out[f"P@{k}"] = yy[:k].mean()
    out["R@100"] = yy[:100].sum() / yy.sum()
    out["first_hit"] = int(np.argmax(yy == 1) + 1)
    top = set(order[:100])
    out["case_hit@100"] = case_members.apply(lambda m: any(r in top for r in m)).mean()
    out["member_recall@100"] = case_members.apply(lambda m: np.mean([r in top for r in m])).mean()
    return out

RESULTS, SCORES = [], {}

def run(name, model, X_source, supervised=True):
    """Score + evaluate on both feature sets x 5 seeds; returns (mean, std) tables."""
    rows = []
    for fs, cols in FEATURE_SETS.items():
        per_seed = []
        for s in SEEDS:
            m = clone(model)
            if "random_state" in m.get_params():
                m.set_params(random_state=s)
            for k in m.get_params():                     # seeds inside the pipeline
                if k.endswith("__random_state"):
                    m.set_params(**{k: s})
            if supervised:
                sc_ = oof_scores(m, X_source[cols], s)
            else:
                sc_ = pd.Series(m.fit(X_source[cols]).score_samples(X_source[cols]) * -1,
                                index=X_source.index)
            per_seed.append(sc_)
            rows.append({"model": name, "features": fs, "seed": s, **evaluate(sc_, s)})
        SCORES[(name, fs)] = pd.concat(per_seed, axis=1).mean(axis=1)
    df = pd.DataFrame(rows)
    RESULTS.append(df)
    g = df.drop(columns=["model", "seed"]).groupby("features", sort=False)
    return g.mean().round(3), g.std().round(3)

# random-ranking baseline
rand_rows = [{"model": "Random", "features": fs, "seed": s,
              **evaluate(pd.Series(np.random.default_rng(100 + s).random(len(y)), index=y.index), s)}
             for fs in FEATURE_SETS for s in SEEDS]
RESULTS.append(pd.DataFrame(rand_rows))
print(f"\nrandom-ranking precision = positive rate = {y.mean():.3f}")

records 1002, positives 159 (15.9%), CV groups among positives 31

T1 log(1+x):
                            skew_before  skew_after_log
n_engagements                      1.72            0.16
open_engagements                   2.62            1.58
active_months                      1.54           -0.61
n_eng_markets                      3.86            0.53
n_addresses                        3.01            2.63
n_institution_links               14.87            3.57
link_serves                       15.30            5.43
link_site_of                       8.59            4.76
public_institution_links           8.16            3.33
under_level_approvals              9.64            6.31
self_approvals                     4.91            4.42
missing_compliance_reviews         3.80            2.91
dd_lead_days                       8.40            0.31
eng_record_lag_days                2.47            0.26

random-ranking precision = positive rate = 0.159


## Cell 8 · Model 1: Linear Regression

### Why start here

When you have a bunch of $x$'s and a $y$, this is **the first model that comes to mind**: write $y$ as a weighted sum of the $x$'s,

$$
\hat y_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \cdots + \beta_p x_{ip}
$$

and estimate $\beta$ with ordinary least squares (OLS). When $y$ is 0/1 this is also called a **linear probability model**: we treat $\hat y_i$ as "how likely this vendor belongs to a labelled case" and use it directly to rank vendors.

(Strictly speaking, *simple* linear regression means a single predictor. Since we use all the variables, this is *multiple* linear regression.)

### Setup

| Item | Setting |
|---|---|
| Input | `X_lin` (with T1 log transform) → T3 imputation + missing indicators → T2 winsorizing → T4 standardization |
| Model | `LinearRegression()`, no regularization |
| Output (risk score) | $\hat y_i$, sorted from high to low |
| Training / evaluation | 5-fold grouped CV × 5 seeds, both feature sets |

### Problems we expect (and the reason to move on to the next model)

1. **$\hat y$ is not bounded to [0, 1]:** a straight line has no limits, so we can get negative "probabilities" or ones above 1. The ranking still works, but the numbers can't be read as probabilities.
2. **The error assumptions don't hold:** $y$ only takes the values 0 and 1, so the residual variance changes with $\hat y$ (heteroskedasticity), and the OLS standard errors and significance tests aren't reliable.
3. **It can only express straight-line relationships** like "bigger is riskier" or "bigger is safer". It can't express thresholds (e.g. "even one occurrence is suspicious") or interactions between variables.

In [4]:
# ============================================================
# Cell 9. model_linear
# ============================================================
from sklearn.linear_model import LinearRegression

model_linear = make_pipeline(*linear_prep(), LinearRegression())
mean_linear, std_linear = run("1 Linear regression", model_linear, X_lin)
print("mean over 5 seeds"); print(mean_linear.to_string())
print("\nstd over 5 seeds"); print(std_linear[["AP", "P@50", "R@100"]].to_string())

# ---- Problem 1: predictions outside [0, 1] ----
cols = FEATURE_SETS["No-shortcut"]
fit_linear = clone(model_linear).fit(X_lin[cols], y)
pred = fit_linear.predict(X_lin[cols])
print(f"\npredictions < 0: {(pred < 0).sum()},  > 1: {(pred > 1).sum()}  "
      f"(range {pred.min():.2f} to {pred.max():.2f})")

# ---- Coefficients (standardized, No-shortcut, fit on all data, for interpretation only) ----
names = list(cols) + [f"missing:{c}" for c in np.array(cols)[fit_linear[0].indicator_.features_]]
coef_linear = pd.Series(fit_linear[-1].coef_, index=names).sort_values()
print("\nlargest coefficients (per 1 SD):")
print(pd.concat([coef_linear.head(5), coef_linear.tail(8)]).round(3).to_string())

mean over 5 seeds
                AP  P@10   P@25   P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100
features                                                                                        
Full         0.589   1.0  0.992  0.796  0.638  0.401        1.0         0.679              0.420
No-shortcut  0.487   1.0  1.000  0.892  0.556  0.350        1.0         0.527              0.386

std over 5 seeds
                AP   P@50  R@100
features                        
Full         0.025  0.048  0.019
No-shortcut  0.020  0.030  0.014

predictions < 0: 149,  > 1: 19  (range -0.21 to 1.32)

largest coefficients (per 1 SD):
link_serves                -0.311
link_site_of               -0.189
link_affiliated_with       -0.188
high_band_share            -0.059
n_engagements              -0.055
expert_hcp_consultant       0.021
dd_enhanced                 0.024
log_total_usd               0.030
activity_density            0.054
sensitive_expense_share     0.054
missing:activit

## Cell 10 · Model 2: Logistic Regression

### Why it makes sense and what it improves

This addresses problems 1 and 2 from Model 1: since $y$ is 0/1, we should model the **probability** directly. Logistic regression still uses a linear combination, but passes it through the sigmoid function so the output stays in (0, 1):

$$
\hat P(y_i = 1 \mid x_i) = \frac{1}{1 + e^{-(\beta_0 + \beta^\top x_i)}}
\qquad\Longleftrightarrow\qquad
\log \frac{P}{1 - P} = \beta_0 + \beta^\top x_i
$$

- The output is always between 0 and 1, so it can be read as a probability.
- It's fitted by maximum likelihood, assuming $y$ is Bernoulli, which matches 0/1 data.
- **The coefficients are still easy to interpret:** $e^{\beta_j}$ is an odds ratio, i.e. how much the odds of "positive vs. not positive" get multiplied when $x_j$ goes up by one standard deviation. This is really useful for writing reason codes for analysts.

### Setup

| Item | Setting | Reason |
|---|---|---|
| Input | Exactly the same as Model 1 | We only change the model, so any improvement comes from the model itself |
| Regularization | L2, `C = 0.1` | 67 variables but only 159 positives; without regularization the coefficients would be unstable. L2 shrinks them towards 0 |
| Class weighting | `class_weight="balanced"` (T5) | Only 15.9% positives |
| Output | $\hat P(y_i = 1)$ | |

### Remaining limitation

Like Model 1, each variable can only have a **monotone, straight-line effect** (on the log-odds scale). It still can't express thresholds or interactions, like "high share of sensitive expenses **and** engagements across many business units".

In [5]:
# ============================================================
# Cell 11. model_logistic
# ============================================================
from sklearn.linear_model import LogisticRegression

model_logistic = make_pipeline(*linear_prep(),
                               LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
mean_logistic, std_logistic = run("2 Logistic regression", model_logistic, X_lin)
print("mean over 5 seeds"); print(mean_logistic.to_string())
print("\nstd over 5 seeds"); print(std_logistic[["AP", "P@50", "R@100"]].to_string())

# ---- Coefficients and odds ratios (No-shortcut, fit on all data, for interpretation only) ----
cols = FEATURE_SETS["No-shortcut"]
fit_logistic = clone(model_logistic).fit(X_lin[cols], y)
names = list(cols) + [f"missing:{c}" for c in np.array(cols)[fit_logistic[0].indicator_.features_]]
coef_logistic = pd.Series(fit_logistic[-1].coef_[0], index=names).sort_values()
tbl = pd.DataFrame({"coef (per 1 SD)": coef_logistic,
                    "odds ratio": np.exp(coef_logistic)}).round(3)
print("\nlargest coefficients:")
print(pd.concat([tbl.head(5), tbl.tail(8)]).to_string())

mean over 5 seeds
                AP  P@10   P@25   P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100
features                                                                                        
Full         0.602   1.0  0.976  0.808  0.660  0.415        1.0         0.691              0.412
No-shortcut  0.462   1.0  1.000  0.812  0.504  0.317        1.0         0.533              0.362

std over 5 seeds
                AP   P@50  R@100
features                        
Full         0.021  0.036  0.018
No-shortcut  0.020  0.036  0.011

largest coefficients:
                         coef (per 1 SD)  odds ratio
system_sponsor_share              -0.477       0.621
high_band_share                   -0.448       0.639
n_engagements                     -0.447       0.639
link_site_of                      -0.394       0.674
missing:amount_cv                 -0.345       0.708
expert_hcp_consultant              0.202       1.224
svc_flag_gov                       0.204       1.2

## Cell 12 · Model 3: Decision Tree

### Why it makes sense and what it improves

The first two models assume each variable's effect is a straight line. But compliance risk often works more like **rules**:

- **Thresholds:** `under_level_approvals` is suspicious as soon as it's ≥ 1, and 1 vs. 5 doesn't make a huge difference. That's not a straight line.
- **Interactions:** some variables only matter when another condition holds. Offering intermediary services is normal on its own, but "intermediary services + lots of sensitive expenses" is a different story.

A decision tree keeps asking "is this variable ≤ some value?" and splits vendors into groups layer by layer. The share of positives in each leaf becomes the risk score. It **finds thresholds and interactions automatically**, and:

- doesn't need log transforms, winsorizing, or standardization (it only cares about order, as explained in Cell 6);
- is **the easiest model to explain to an analyst**: every vendor's score corresponds to a clear "if ... and ... then ..." path.

### Setup

| Item | Setting | Reason |
|---|---|---|
| Input | Raw `X` (no log/winsorizing/scaling), only T3 imputation | Trees are not affected by monotone transforms |
| Max depth | `max_depth = 4` | At most 4 levels of rules so a person can read it; deeper trees would just memorize the 159 positives in the practice set (overfitting) |
| Min samples per leaf | `min_samples_leaf = 15` | Every rule must cover at least 15 vendors, so we don't create rules for one or two specific vendors |
| Class weighting | `class_weight="balanced"` | Same as T5 |
| Output | (Weighted) share of positives in the vendor's leaf | |

### Limitations we expect

A single tree is **unstable**: if the data changes a little, the first split can change and then the whole tree changes. Also, the scores only take about a dozen distinct values (one per leaf), so the ranking has a lot of ties.

In [6]:
# ============================================================
# Cell 13. model_tree
# ============================================================
from sklearn.tree import DecisionTreeClassifier, export_text

model_tree = make_pipeline(*tree_prep(),
                           DecisionTreeClassifier(max_depth=4, min_samples_leaf=15,
                                                  class_weight="balanced", random_state=0))
mean_tree, std_tree = run("3 Decision tree", model_tree, X)
print("mean over 5 seeds"); print(mean_tree.to_string())
print("\nstd over 5 seeds"); print(std_tree[["AP", "P@50", "R@100"]].to_string())

# ---- Tree rules (No-shortcut, fit on all data, for interpretation only) ----
cols = FEATURE_SETS["No-shortcut"]
fit_tree = clone(model_tree).fit(X[cols], y)
names = list(cols) + [f"missing:{c}" for c in np.array(cols)[fit_tree[0].indicator_.features_]]
print("\ntree rules (class 1 = belongs to a labelled case):")
print(export_text(fit_tree[-1], feature_names=names, show_weights=False, decimals=2))
print(f"distinct score values: {len(np.unique(fit_tree.predict_proba(X[cols])[:, 1]))}")

mean over 5 seeds
                AP  P@10   P@25  P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100
features                                                                                       
Full         0.419  0.72  0.592   0.6  0.562  0.353        1.0         0.558              0.374
No-shortcut  0.467  0.96  0.968   0.9  0.558  0.351        1.0         0.485              0.315

std over 5 seeds
                AP   P@50  R@100
features                        
Full         0.011  0.040  0.023
No-shortcut  0.008  0.037  0.023

tree rules (class 1 = belongs to a labelled case):
|--- under_level_approvals <= 0.50
|   |--- sensitive_expense_share <= 0.79
|   |   |--- log_total_usd <= 14.77
|   |   |   |--- activity_density <= 0.09
|   |   |   |   |--- class: 0
|   |   |   |--- activity_density >  0.09
|   |   |   |   |--- class: 0
|   |   |--- log_total_usd >  14.77
|   |   |   |--- active_months <= 51.50
|   |   |   |   |--- class: 0
|   |   |   |--- active_months >  

## Cell 14 · Model 4: Random Forest

### Why it makes sense and what it improves

This addresses the **instability** of a single tree. We train many trees, and each tree

1. is trained on a different **bootstrap sample** (sampling with replacement), and
2. at each split only considers a **random subset of variables** (`max_features="sqrt"`, about $\sqrt{67} \approx 8$).

Then we average all the trees' predictions. Each tree has its own errors, but averaging reduces the variance a lot, so the result is much more stable. We still keep the tree's ability to capture thresholds and interactions, and the scores become continuous so there are far fewer ties.

### Setup

| Item | Setting | Reason |
|---|---|---|
| Input | Raw `X`, only T3 imputation | Same as the decision tree |
| Number of trees | `n_estimators = 500` | Enough that results no longer change noticeably with more trees |
| Min samples per leaf | `min_samples_leaf = 3` | Individual trees can grow deeper than in Model 3; averaging takes care of the overfitting |
| Candidate variables per split | `max_features = "sqrt"` | Makes the trees different from each other, which is what makes averaging work |
| Class weighting | `class_weight = "balanced_subsample"` | Each tree recomputes weights for its own bootstrap sample |

### Cost

**We lose interpretability:** we can't read 500 trees rule by rule, only look at feature importances. This is exactly the trade-off the dossier cares about: *"the point at which additional model complexity stops earning its cost"*, i.e. whether the extra performance from a more complex model is worth the interpretability we give up.

In [7]:
# ============================================================
# Cell 15. model_rf
# ============================================================
from sklearn.ensemble import RandomForestClassifier

model_rf = make_pipeline(*tree_prep(),
                         RandomForestClassifier(n_estimators=500, min_samples_leaf=3,
                                                max_features="sqrt",
                                                class_weight="balanced_subsample",
                                                n_jobs=-1, random_state=0))
mean_rf, std_rf = run("4 Random forest", model_rf, X)
print("mean over 5 seeds"); print(mean_rf.to_string())
print("\nstd over 5 seeds"); print(std_rf[["AP", "P@50", "R@100"]].to_string())

# ---- Feature importances (No-shortcut, fit on all data) ----
# Note: impurity-based importances favor continuous variables with many values; treat as rough
cols = FEATURE_SETS["No-shortcut"]
fit_rf = clone(model_rf).fit(X[cols], y)
names = list(cols) + [f"missing:{c}" for c in np.array(cols)[fit_rf[0].indicator_.features_]]
imp_rf = pd.Series(fit_rf[-1].feature_importances_, index=names).sort_values(ascending=False)
print("\ntop 12 feature importances:"); print(imp_rf.head(12).round(3).to_string())

mean over 5 seeds
                AP  P@10   P@25   P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100
features                                                                                        
Full         0.578  0.76  0.704  0.704  0.684  0.430        1.2         0.691              0.484
No-shortcut  0.510  0.88  0.936  0.904  0.600  0.377        1.4         0.515              0.373

std over 5 seeds
                AP   P@50  R@100
features                        
Full         0.014  0.052  0.020
No-shortcut  0.016  0.017  0.013



top 12 feature importances:
under_level_approvals    0.079
log_total_usd            0.077
activity_density         0.071
amount_cv                0.051
non_po_invoice_share     0.050
active_months            0.049
round_amount_share       0.047
approval_lead_days       0.044
invoices_per_payment     0.042
dd_lead_days             0.039
eng_record_lag_days      0.034
manual_payment_share     0.032


## Cell 16 · Model 5: Isolation Forest (unsupervised)

### Why we need it

Models 1–4 all need $y$. But:

- the main dataset (~20,000 records) **has no labels**;
- the practice set only has 33 cases, and the dossier explicitly says the main dataset **contains patterns the practice set doesn't**, so what we learn from the practice set may not transfer.

So we need a model that **never looks at $y$** and just asks "which vendors look different from most vendors?". We only use $y$ **afterwards** to check whether the anomalies it finds overlap with the labelled cases.

### How it works

Pick a random variable and a random split point within its range, cut the data in two, and keep going until every point is isolated. **Anomalous points are easier to isolate**, so they need fewer cuts. Averaging the path length over many random trees, a shorter path means a higher anomaly score.

### Setup

| Item | Setting | Reason |
|---|---|---|
| Input | `X_lin` with T3 imputation and T4 standardization, no winsorizing | Winsorizing would pull extreme values back in, which would erase exactly the anomalies we're looking for |
| Number of trees | `n_estimators = 500` | |
| Training | All data, **no $y$, no cross-validation needed** | It's unsupervised, so there's no "seeing the answer" problem |
| Output | Anomaly score (higher = more anomalous) | As the dossier reminds us, this is a **ranking**, not a probability |

### What we expect

"Anomalous" doesn't mean "risky". Many real case members look completely normal on their own row (the problem is in how they are connected), so we expect the isolation forest to do clearly worse than the supervised models in Layer 1. But it's the **only method that doesn't depend on the practice labels**, and it might improve once Layers 2 and 3 add relational variables.

In [8]:
# ============================================================
# Cell 17. model_iforest
# ============================================================
from sklearn.ensemble import IsolationForest

model_iforest = make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                              StandardScaler(),
                              IsolationForest(n_estimators=500, random_state=0))
mean_if, std_if = run("5 Isolation forest", model_iforest, X_lin, supervised=False)
print("mean over 5 seeds"); print(mean_if.to_string())
print("\nstd over 5 seeds"); print(std_if[["AP", "P@50", "R@100"]].to_string())

mean over 5 seeds
                AP  P@10   P@25   P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100
features                                                                                        
Full         0.195   0.0  0.064  0.200  0.246  0.155       21.0         0.509              0.247
No-shortcut  0.182   0.0  0.096  0.184  0.214  0.135       20.4         0.485              0.243

std over 5 seeds
                AP   P@50  R@100
features                        
Full         0.004  0.032  0.012
No-shortcut  0.004  0.017  0.007


## Cell 18 · Model 6: Multi-level Scorecard (`model_ms`)

### What this is

This is the **traditional scoring approach** we originally had in mind: instead of letting a model learn a formula freely, we bin each variable using fixed rules, give each bin some points, and add up a weighted total.

$$
\text{Total}_i \;=\; \sum_{j=1}^{p} w_j \cdot \text{Subscore}_{ij},
\qquad \text{Subscore}_{ij} \in [0, 10],
\qquad \sum_j w_j = 10
\;\;\Rightarrow\;\; \text{Total}_i \in [0, 100]
$$

There are three steps:

1. **Binning:** split each variable into a few bins using thresholds;
2. **Points within a variable:** give each bin a subscore between 0 and 10;
3. **Weights across variables:** give each variable a weight $w_j$.

The difference from Models 1–4 is that variables don't enter the formula as raw numbers; they are first turned into "which bin". The structure is fixed, so an analyst can directly read something like "this vendor scored 10 on approval violations and 7.5 on payment activity...".

### How many bins per variable (rules)

The number of bins depends on what each variable can actually support, so we can't just use 4 or 5 bins for everything:

| Variable type | How we decide | Bins | How thresholds are set |
|---|---|---|---|
| **Binary** | Only two distinct values | 2 | Between the two values |
| **Concentrated** | The most common value covers ≥ 60% of vendors (e.g. 96% of vendors have `under_level_approvals` = 0) | 3 | ① the most common value; ② the median of the values above it. So: "typical / a bit above typical / well above typical" |
| **Continuous** | Everything else | 5 | Quintiles (20%, 40%, 60%, 80%). If repeated values make cutoffs coincide, they get merged, so a variable can end up with fewer than 5 bins |

- **Direction:** some variables are riskier when they're *smaller* (e.g. number of engagements). We use the sign of the rank correlation between bin index and $y$ in the training data, so that 10 points always go to the riskiest bin.
- **Missing values:** imputed with the training median before binning (same as T3 in Cell 6).
- **No log or scaling needed:** binning only depends on the order of values.
- **No leakage:** in cross-validation, thresholds, directions, bin points, and weights are all computed **on the training fold only**. The tables shown below use all 1,002 vendors and are just for illustration.

### Points within a variable: two options

| Option | Subscore | Notes |
|---|---|---|
| **Equal spacing** (`points="equal"`) | Bin $k$ out of $K$ ($k = 0, \dots, K-1$) gets $10 \cdot k/(K-1)$: 2 bins → 0/10, 3 bins → 0/5/10, 5 bins → 0/2.5/5/7.5/10 | Our original idea. The lowest bin gets 0 ("no warning sign"). Compared with our first draft of 2/4/6/8/10, each variable only shifts by a constant that is the same for everyone, so **the ranking is identical** |
| **By positive rate** (`points="rate"`) | Each bin's (smoothed) share of positives, rescaled to 0–10: the bin with the lowest rate gets 0 and the highest gets 10 | Doesn't assume risk goes up evenly from bin to bin. For example, it can capture "going from 0 to 1 is a big jump, but 1 to 5 isn't" |

Cell 19 below lists the number of bins and the thresholds for every variable.

In [9]:
# ============================================================
# Cell 19. Model 6 structure: number of bins and thresholds for each variable
# ============================================================
from sklearn.base import ClassifierMixin
from scipy.optimize import minimize
from scipy.stats import spearmanr

TYPE_LABEL = {"binary": "binary", "concentrated": "concentrated", "continuous": "continuous"}

def fit_nonneg_logistic(S, yv, C=1.0):
    """Logistic regression with weights >= 0 (L2 penalty + balanced class weights). S: subscore matrix (0-10)."""
    Z = S / 10.0
    n, J = Z.shape
    npos = yv.sum()
    sw = np.where(yv == 1, n / (2 * npos), n / (2 * (n - npos)))
    def f(theta):
        b0, w = theta[0], theta[1:]
        z = b0 + Z @ w
        loss = C * np.sum(sw * (np.logaddexp(0, z) - yv * z)) + 0.5 * w @ w
        r = sw * (1 / (1 + np.exp(-z)) - yv)
        return loss, np.r_[C * r.sum(), C * Z.T @ r + w]
    res = minimize(f, np.zeros(J + 1), jac=True, method="L-BFGS-B",
                   bounds=[(None, None)] + [(0, None)] * J)
    return res.x[1:], res.x[0]

class MultiLevelScorecard(BaseEstimator, ClassifierMixin):
    """
    Multi-level scorecard:
      1. Bin each variable by type (binary: 2 bins; concentrated: 3; continuous: 5 by quantiles)
      2. Give each bin a 0-10 subscore: points="equal" equally spaced; points="rate" by the bin's positive rate
      3. Total = sum_j w_j * subscore_j with sum_j w_j = 10, so the total is 0-100
         weights="equal": same weight for every variable; weights="learned": non-negative logistic regression
    Thresholds, directions, subscores and weights all come from the data passed to fit (= training fold in CV).
    """
    def __init__(self, points="equal", weights="learned", C=1.0,
                 concentrated_share=0.6, n_quantile_bins=5, smoothing=5.0):
        self.points, self.weights, self.C = points, weights, C
        self.concentrated_share, self.n_quantile_bins = concentrated_share, n_quantile_bins
        self.smoothing = smoothing

    def _cuts(self, s):
        u = np.unique(s)
        if len(u) <= 2:
            return "binary", [float(u[0])] if len(u) == 2 else []
        vals, cnt = np.unique(s, return_counts=True)
        mode, share = vals[cnt.argmax()], cnt.max() / len(s)
        if share >= self.concentrated_share:
            above = s[s > mode]
            if len(np.unique(above)) <= 1:
                return "concentrated", [float(mode)]
            q = np.quantile(above, 0.5)
            if q >= above.max():
                q = np.unique(above)[-2]
            return "concentrated", [float(mode), float(q)]
        qs = np.quantile(s, np.linspace(0, 1, self.n_quantile_bins + 1)[1:-1])
        cuts = sorted(set(float(q) for q in qs if q < s.max()))
        return "continuous", cuts

    def fit(self, Xd, yd):
        Xd = pd.DataFrame(Xd)
        yv = np.asarray(yd).astype(float)
        self.columns_ = list(Xd.columns)
        self.median_ = Xd.median()
        Xf = Xd.fillna(self.median_)
        p = yv.mean()
        self.bins_ = {}
        for c in self.columns_:
            s = Xf[c].to_numpy(float)
            kind, cuts = self._cuts(s)
            b = np.searchsorted(cuts, s, side="left")
            K = len(cuts) + 1
            n = np.bincount(b, minlength=K)
            pos = np.bincount(b, weights=yv, minlength=K)
            rate = (pos + self.smoothing * p) / (n + self.smoothing)    # smoothed positive rate
            rho = spearmanr(b, yv).correlation if K > 1 else 0.0
            ascending = not (rho < 0)                                      # nan -> treat as ascending
            if K == 1:
                pts = np.zeros(1)
            elif self.points == "equal":
                order = np.arange(K) if ascending else np.arange(K)[::-1]
                pts = 10.0 * order / (K - 1)
            else:
                rng_ = rate.max() - rate.min()
                pts = 10.0 * (rate - rate.min()) / rng_ if rng_ > 0 else np.zeros(K)
            self.bins_[c] = dict(kind=kind, cuts=cuts, points=pts, n=n, rate=pos / np.maximum(n, 1),
                                 ascending=ascending)
        S = self.subscores(Xd).to_numpy()
        if self.weights == "equal":
            w = np.ones(S.shape[1])
        else:
            w, _ = fit_nonneg_logistic(S, yv, self.C)
        self.raw_weights_ = pd.Series(w, index=self.columns_)
        total = w.sum()
        self.weights_ = (self.raw_weights_ * 10 / total if total > 0
                         else pd.Series(10 / len(w), index=self.columns_))
        self.classes_ = np.array([0, 1])
        return self

    def subscores(self, Xd):
        Xf = pd.DataFrame(Xd)[self.columns_].fillna(self.median_)
        return pd.DataFrame({c: self.bins_[c]["points"][np.searchsorted(self.bins_[c]["cuts"],
                                                                        Xf[c].to_numpy(float), side="left")]
                             for c in self.columns_}, index=Xf.index)

    def total_score(self, Xd):
        return self.subscores(Xd) @ self.weights_          # 0-100

    def predict_proba(self, Xd):
        t = self.total_score(Xd).to_numpy() / 100.0
        return np.c_[1 - t, t]

def bin_labels(cuts):
    fmt = lambda v: f"{v:.3g}"
    if not cuts:
        return ["all"]
    labs = [f"≤ {fmt(cuts[0])}"]
    labs += [f"({fmt(a)}, {fmt(b)}]" for a, b in zip(cuts[:-1], cuts[1:])]
    labs += [f"> {fmt(cuts[-1])}"]
    return labs

def structure_table(sc):
    rows = []
    for c in sc.columns_:
        b = sc.bins_[c]
        rows.append({"variable": c, "type": TYPE_LABEL[b["kind"]], "n_bins": len(b["cuts"]) + 1,
                     "thresholds": ", ".join(f"{v:.3g}" for v in b["cuts"]),
                     "direction": "higher = riskier" if b["ascending"] else "lower = riskier"})
    return pd.DataFrame(rows).set_index("variable")


cols_ns = FEATURE_SETS["No-shortcut"]
sc_struct = MultiLevelScorecard(points="equal", weights="equal").fit(X[cols_ns], y)
struct = structure_table(sc_struct)
print(f"{len(struct)} variables (No-shortcut set)")
print(struct["type"].value_counts().to_string())
print(struct["n_bins"].value_counts().sort_index().rename("n_variables").to_string())
struct

60 variables (No-shortcut set)
type
binary          26
concentrated    18
continuous      16
n_bins
2    27
3    19
4     2
5    12


,type,n_bins,thresholds,direction
variable,,,,
is_individual,binary,2,0,lower = riskier
expert_hcp_consultant,binary,2,0,higher = riskier
expert_investigator_kol,binary,2,0,lower = riskier
expert_regulatory_access,binary,2,0,lower = riskier
expert_other,binary,2,0,lower = riskier
has_lei,binary,2,0,lower = riskier
has_registry_no,binary,2,0,higher = riskier
remit_country_mismatch,binary,2,0,higher = riskier
migrated_record,binary,2,0,lower = riskier


## Cell 20 · Bins and Points for Each Variable

The table below is computed on all 1,002 vendors (in cross-validation each training fold recomputes it, so the thresholds differ slightly).

**How to read it:** each bin is written as `range → equal-spacing points / positive-rate points (number of vendors, share of positives)`. For example, the `under_level_approvals` row (number of approvals signed by an approver below the required level): `≤ 0` gets **0** points (equal spacing) or 0.0 (by rate), with 964 vendors in the bin and 13% positives; `(0, 1]` gets **5** points (equal spacing) or 10.0 (by rate), with 23 vendors in the bin and 100% positives; `> 1` gets **10** points (equal spacing) or 9.2 (by rate), with 15 vendors in the bin and 100% positives. Notice that with equal spacing, a vendor with just one violation only gets half the points, even though every vendor in that bin is a positive; the rate-based version doesn't have this problem.

The share of positives in each bin tells us right away whether a variable is useful: if every bin is close to the overall 16%, the variable won't help no matter how many points we give it.

| Variable | Meaning | Type | Bins | Direction | Bins: range → equal pts / rate pts (n, positive share) |
|---|---|---|---|---|---|
| `is_individual` | Individual expert (vs. organization) | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=863, 17% pos.)<br>> 0 → **0** / 0.0 (n=139, 9% pos.) |
| `expert_hcp_consultant` | Expert type: HCP consultant | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=934, 16% pos.)<br>> 0 → **10** / 10.0 (n=68, 18% pos.) |
| `expert_investigator_kol` | Expert type: investigator / KOL | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=967, 16% pos.)<br>> 0 → **0** / 0.0 (n=35, 3% pos.) |
| `expert_regulatory_access` | Expert type: regulatory / market access | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=990, 16% pos.)<br>> 0 → **0** / 0.0 (n=12, 0% pos.) |
| `expert_other` | Expert type: other (3 rare types merged) | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=978, 16% pos.)<br>> 0 → **0** / 0.0 (n=24, 0% pos.) |
| `has_lei` | Has an LEI | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=674, 16% pos.)<br>> 0 → **0** / 0.0 (n=328, 15% pos.) |
| `has_registry_no` | Has a company registry number | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=638, 15% pos.)<br>> 0 → **10** / 10.0 (n=364, 17% pos.) |
| `remit_country_mismatch` | Remit-to country differs from operating country | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=974, 16% pos.)<br>> 0 → **10** / 10.0 (n=28, 25% pos.) |
| `migrated_record` | Record re-created in the 2022 ERP migration | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=833, 16% pos.)<br>> 0 → **0** / 0.0 (n=169, 14% pos.) |
| `onboarded_in_event_window` | Onboarded during an acquisition or vaccine-launch window | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=868, 16% pos.)<br>> 0 → **0** / 0.0 (n=134, 13% pos.) |
| `has_declared_group` | Declared a parent group | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=914, 16% pos.)<br>> 0 → **10** / 10.0 (n=88, 19% pos.) |
| `public_email` | Uses a public email provider | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=893, 16% pos.)<br>> 0 → **0** / 0.0 (n=109, 11% pos.) |
| `q_public_official_interaction` | Interacts with public officials on our behalf | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=874, 14% pos.)<br>> 0 → **10** / 10.0 (n=128, 28% pos.) |
| `dd_sensitive_answer` | Self-reported state-owned / PEP-connected / former official (any) | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=990, 16% pos.)<br>> 0 → **0** / 0.0 (n=12, 8% pos.) |
| `adverse_media` | Adverse media flag | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=980, 16% pos.)<br>> 0 → **10** / 10.0 (n=22, 23% pos.) |
| `screening_potential_match` | Screening potential match (cleared) | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=962, 16% pos.)<br>> 0 → **0** / 0.0 (n=40, 12% pos.) |
| `dd_enhanced` | Enhanced due diligence | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=855, 14% pos.)<br>> 0 → **10** / 10.0 (n=147, 27% pos.) |
| `dd_lead_days` | Days from DD completion to onboarding | continuous | 5 | higher = riskier | ≤ 16 → **0** / 5.0 (n=220, 16% pos.)<br>(16, 27] → **2.5** / 0.0 (n=191, 14% pos.)<br>(27, 39] → **5** / 6.2 (n=202, 16% pos.)<br>(39, 51] → **7.5** / 10.0 (n=198, 18% pos.)<br>> 51 → **10** / 3.0 (n=191, 15% pos.) |
| `n_service_parents` | Number of service parent categories | concentrated | 3 | lower = riskier | ≤ 1 → **10** / 3.7 (n=880, 16% pos.)<br>(1, 2] → **5** / 0.0 (n=120, 11% pos.)<br>> 2 → **0** / 10.0 (n=2, 50% pos.) |
| `svc_flag_hcp` | Provides HCP-facing services | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=475, 16% pos.)<br>> 0 → **0** / 0.0 (n=527, 16% pos.) |
| `svc_flag_gov` | Provides government-facing services | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=652, 14% pos.)<br>> 0 → **10** / 10.0 (n=350, 19% pos.) |
| `svc_flag_int` | Provides intermediary services | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=826, 15% pos.)<br>> 0 → **10** / 10.0 (n=176, 22% pos.) |
| `svc_flag_flow` | Provides high cash/goods flow services | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=724, 15% pos.)<br>> 0 → **10** / 10.0 (n=278, 19% pos.) |
| `n_engagements` | Number of engagements | continuous | 5 | lower = riskier | ≤ 1 → **10** / 9.9 (n=282, 18% pos.)<br>(1, 2] → **7.5** / 5.2 (n=285, 15% pos.)<br>(2, 3] → **5** / 0.0 (n=190, 12% pos.)<br>(3, 4] → **2.5** / 10.0 (n=129, 18% pos.)<br>> 4 → **0** / 9.0 (n=116, 17% pos.) |
| `n_eng_markets` | Number of engagement markets | concentrated | 3 | higher = riskier | ≤ 1 → **0** / 0.0 (n=909, 16% pos.)<br>(1, 2] → **5** / 1.1 (n=88, 17% pos.)<br>> 2 → **10** / 10.0 (n=5, 40% pos.) |
| `n_business_units` | Number of business units | continuous | 3 | higher = riskier | ≤ 1 → **0** / 0.0 (n=612, 12% pos.)<br>(1, 2] → **5** / 5.3 (n=295, 20% pos.)<br>> 2 → **10** / 10.0 (n=95, 28% pos.) |
| `n_contract_types` | Number of contract types | concentrated | 3 | lower = riskier | ≤ 1 → **10** / 1.3 (n=951, 16% pos.)<br>(1, 2] → **5** / 0.0 (n=49, 14% pos.)<br>> 2 → **0** / 10.0 (n=2, 50% pos.) |
| `high_band_share` | Share of high-value (band 3-4) engagements | continuous | 4 | lower = riskier | ≤ 0 → **10** / 3.8 (n=590, 16% pos.)<br>(0, 0.2] → **6.66667** / 5.4 (n=12, 17% pos.)<br>(0.2, 0.8] → **3.33333** / 10.0 (n=200, 19% pos.)<br>> 0.8 → **0** / 0.0 (n=200, 14% pos.) |
| `missing_sponsor_share` | Share of engagements with no sponsor | concentrated | 3 | lower = riskier | ≤ 0 → **10** / 10.0 (n=877, 16% pos.)<br>(0, 0.5] → **5** / 8.9 (n=93, 15% pos.)<br>> 0.5 → **0** / 0.0 (n=32, 3% pos.) |
| `system_sponsor_share` | Share of engagements with the system-default sponsor | concentrated | 3 | lower = riskier | ≤ 0 → **10** / 10.0 (n=962, 16% pos.)<br>(0, 0.333] → **5** / 1.0 (n=27, 4% pos.)<br>> 0.333 → **0** / 0.0 (n=13, 0% pos.) |
| `open_engagements` | Engagements still open at t | concentrated | 3 | higher = riskier | ≤ 0 → **0** / 0.0 (n=728, 13% pos.)<br>(0, 2] → **5** / 8.2 (n=169, 22% pos.)<br>> 2 → **10** / 10.0 (n=105, 25% pos.) |
| `eng_record_lag_days` | Avg. days from engagement start to being recorded | continuous | 5 | higher = riskier | ≤ 1.25 → **0** / 7.3 (n=202, 17% pos.)<br>(1.25, 2] → **2.5** / 0.0 (n=219, 13% pos.)<br>(2, 3] → **5** / 10.0 (n=229, 18% pos.)<br>(3, 4.5] → **7.5** / 1.5 (n=162, 14% pos.)<br>> 4.5 → **10** / 8.2 (n=190, 17% pos.) |
| `missing_approver_share` | Share of financial approvals with no approver | concentrated | 3 | lower = riskier | ≤ 0 → **10** / 10.0 (n=754, 17% pos.)<br>(0, 0.5] → **5** / 10.0 (n=157, 17% pos.)<br>> 0.5 → **0** / 0.0 (n=91, 3% pos.) |
| `under_level_approvals` | Approvals signed below the required level | concentrated | 3 | higher = riskier | ≤ 0 → **0** / 0.0 (n=964, 13% pos.)<br>(0, 1] → **5** / 10.0 (n=23, 100% pos.)<br>> 1 → **10** / 9.2 (n=15, 100% pos.) |
| `self_approvals` | Approvals signed by the sponsor themselves | concentrated | 3 | higher = riskier | ≤ 0 → **0** / 6.7 (n=952, 16% pos.)<br>(0, 1] → **5** / 10.0 (n=44, 20% pos.)<br>> 1 → **10** / 0.0 (n=6, 0% pos.) |
| `missing_compliance_reviews` | Engagements missing a required compliance review | concentrated | 3 | lower = riskier | ≤ 0 → **10** / 10.0 (n=885, 17% pos.)<br>(0, 2] → **5** / 4.3 (n=87, 11% pos.)<br>> 2 → **0** / 0.0 (n=30, 7% pos.) |
| `approval_lead_days` | Avg. days from approval to engagement start | continuous | 5 | higher = riskier | ≤ 12.5 → **0** / 7.3 (n=208, 18% pos.)<br>(12.5, 16] → **2.5** / 0.0 (n=205, 11% pos.)<br>(16, 18.5] → **5** / 5.1 (n=192, 16% pos.)<br>(18.5, 22] → **7.5** / 10.0 (n=196, 20% pos.)<br>> 22 → **10** / 4.3 (n=201, 15% pos.) |
| `log_total_usd` | Total payments (log USD) | continuous | 5 | higher = riskier | ≤ 10.2 → **0** / 0.0 (n=201, 9% pos.)<br>(10.2, 12.1] → **2.5** / 7.2 (n=200, 18% pos.)<br>(12.1, 13.4] → **5** / 10.0 (n=200, 22% pos.)<br>(13.4, 14.9] → **7.5** / 10.0 (n=200, 22% pos.)<br>> 14.9 → **10** / 0.4 (n=201, 9% pos.) |
| `active_months` | Months with payments | continuous | 5 | lower = riskier | ≤ 7 → **10** / 1.2 (n=226, 14% pos.)<br>(7, 13] → **7.5** / 10.0 (n=183, 23% pos.)<br>(13, 20] → **5** / 0.0 (n=193, 13% pos.)<br>(20, 33] → **2.5** / 2.9 (n=202, 16% pos.)<br>> 33 → **0** / 1.2 (n=198, 14% pos.) |
| `activity_density` | Months with payments / months since onboarding | continuous | 5 | higher = riskier | ≤ 0.121 → **0** / 0.0 (n=201, 7% pos.)<br>(0.121, 0.25] → **2.5** / 5.4 (n=200, 16% pos.)<br>(0.25, 0.469] → **5** / 4.1 (n=200, 14% pos.)<br>(0.469, 0.89] → **7.5** / 6.9 (n=200, 18% pos.)<br>> 0.89 → **10** / 10.0 (n=201, 23% pos.) |
| `amount_cv` | Coefficient of variation of monthly payments | continuous | 5 | higher = riskier | ≤ 0.362 → **0** / 0.6 (n=201, 13% pos.)<br>(0.362, 0.464] → **2.5** / 10.0 (n=200, 20% pos.)<br>(0.464, 0.578] → **5** / 1.3 (n=200, 14% pos.)<br>(0.578, 0.77] → **7.5** / 0.0 (n=200, 13% pos.)<br>> 0.77 → **10** / 7.2 (n=201, 18% pos.) |
| `round_amount_share` | Share of round-amount invoices | continuous | 5 | higher = riskier | ≤ 0.0637 → **0** / 3.0 (n=201, 15% pos.)<br>(0.0637, 0.0719] → **2.5** / 9.4 (n=200, 19% pos.)<br>(0.0719, 0.0788] → **5** / 0.0 (n=200, 13% pos.)<br>(0.0788, 0.0885] → **7.5** / 0.0 (n=200, 13% pos.)<br>> 0.0885 → **10** / 10.0 (n=201, 19% pos.) |
| `manual_payment_share` | Share of manual payments | continuous | 5 | higher = riskier | ≤ 0 → **0** / 8.3 (n=387, 17% pos.)<br>(0, 0.00732] → **2.5** / 4.3 (n=14, 14% pos.)<br>(0.00732, 0.0189] → **5** / 0.0 (n=201, 12% pos.)<br>(0.0189, 0.0317] → **7.5** / 6.5 (n=202, 16% pos.)<br>> 0.0317 → **10** / 10.0 (n=198, 18% pos.) |
| `invoices_per_payment` | Invoices per payment | continuous | 5 | higher = riskier | ≤ 1.04 → **0** / 7.1 (n=201, 16% pos.)<br>(1.04, 1.08] → **2.5** / 0.0 (n=200, 14% pos.)<br>(1.08, 1.1] → **5** / 2.9 (n=200, 15% pos.)<br>(1.1, 1.13] → **7.5** / 7.3 (n=200, 16% pos.)<br>> 1.13 → **10** / 10.0 (n=201, 17% pos.) |
| `non_po_invoice_share` | Share of non-PO invoices | continuous | 5 | lower = riskier | ≤ 0.115 → **10** / 10.0 (n=201, 21% pos.)<br>(0.115, 0.163] → **7.5** / 6.1 (n=200, 17% pos.)<br>(0.163, 0.2] → **5** / 2.6 (n=229, 14% pos.)<br>(0.2, 0.299] → **2.5** / 6.6 (n=171, 18% pos.)<br>> 0.299 → **0** / 0.0 (n=201, 11% pos.) |
| `p_card_share` | Share of p-card payments | continuous | 4 | higher = riskier | ≤ 0 → **0** / 0.0 (n=545, 15% pos.)<br>(0, 0.02] → **3.33333** / 10.0 (n=58, 19% pos.)<br>(0.02, 0.0484] → **6.66667** / 0.5 (n=199, 16% pos.)<br>> 0.0484 → **10** / 3.2 (n=200, 16% pos.) |
| `expense_reimbursement_share` | Share of expense reimbursements | concentrated | 3 | lower = riskier | ≤ 0 → **10** / 10.0 (n=919, 17% pos.)<br>(0, 0.143] → **5** / 0.0 (n=45, 4% pos.)<br>> 0.143 → **0** / 2.9 (n=38, 8% pos.) |
| `n_expense_classes` | Number of expense classifications | concentrated | 3 | higher = riskier | ≤ 1 → **0** / 0.0 (n=942, 15% pos.)<br>(1, 2] → **5** / 7.4 (n=57, 21% pos.)<br>> 2 → **10** / 10.0 (n=3, 33% pos.) |
| `tov_hcp_share` | Share of HCP/HCO transfer-of-value payments | continuous | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=545, 16% pos.)<br>> 0 → **10** / 10.0 (n=457, 16% pos.) |
| `sensitive_expense_share` | Share of sensitive expenses (commissions/grants/sponsorships/gov. fees) | concentrated | 3 | higher = riskier | ≤ 0 → **0** / 0.0 (n=890, 13% pos.)<br>(0, 0.915] → **5** / 3.5 (n=12, 25% pos.)<br>> 0.915 → **10** / 10.0 (n=100, 41% pos.) |
| `n_addresses` | Number of address records | concentrated | 3 | higher = riskier | ≤ 1 → **0** / 0.2 (n=914, 16% pos.)<br>(1, 2] → **5** / 0.0 (n=86, 15% pos.)<br>> 2 → **10** / 10.0 (n=2, 100% pos.) |
| `risky_address_type` | Used a registered-agent / virtual-office / PO-box / coworking address | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=919, 16% pos.)<br>> 0 → **0** / 0.0 (n=83, 16% pos.) |
| `org_at_residential` | Organization registered at a residential address | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=975, 16% pos.)<br>> 0 → **0** / 0.0 (n=27, 15% pos.) |
| `address_type_missing` | Address type not classified | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=598, 15% pos.)<br>> 0 → **10** / 10.0 (n=404, 17% pos.) |
| `n_institution_links` | Number of institution links | concentrated | 3 | higher = riskier | ≤ 0 → **0** / 1.7 (n=829, 15% pos.)<br>(0, 1] → **5** / 0.0 (n=117, 9% pos.)<br>> 1 → **10** / 10.0 (n=56, 46% pos.) |
| `public_institution_links` | Links to public-sector institutions | concentrated | 3 | higher = riskier | ≤ 0 → **0** / 0.0 (n=868, 15% pos.)<br>(0, 1] → **5** / 0.4 (n=106, 16% pos.)<br>> 1 → **10** / 10.0 (n=28, 50% pos.) |
| `link_serves` | Institutions served (distributor) | concentrated | 3 | higher = riskier | ≤ 0 → **0** / 0.0 (n=947, 14% pos.)<br>(0, 2] → **5** / 10.0 (n=32, 47% pos.)<br>> 2 → **10** / 8.5 (n=23, 43% pos.) |
| `link_site_of` | Institutions as trial site | concentrated | 3 | lower = riskier | ≤ 0 → **10** / 5.0 (n=952, 16% pos.)<br>(0, 1] → **5** / 0.0 (n=49, 2% pos.)<br>> 1 → **0** / 10.0 (n=1, 100% pos.) |
| `link_affiliated_with` | Institutions the expert is affiliated with | binary | 2 | lower = riskier | ≤ 0 → **10** / 10.0 (n=934, 16% pos.)<br>> 0 → **0** / 0.0 (n=68, 13% pos.) |
| `country_bo_access_low` | Country has low beneficial-ownership register access | binary | 2 | higher = riskier | ≤ 0 → **0** / 0.0 (n=365, 15% pos.)<br>> 0 → **10** / 10.0 (n=637, 16% pos.) |

In [10]:
# ============================================================
# Cell 21. Weight calculation
#   (1) Equal weights: w = 10 / 60 for every variable
#   (2) Learned weights: non-negative logistic regression (subscores -> y), rescaled to sum to 10
#   We also compare each variable's actual influence on the ranking = w_j x SD(subscore_j)
# ============================================================
weight_tables = {}
for pts in ["equal", "rate"]:
    sc_e = MultiLevelScorecard(points=pts, weights="equal").fit(X[cols_ns], y)
    sc_l = MultiLevelScorecard(points=pts, weights="learned").fit(X[cols_ns], y)
    sd = sc_e.subscores(X[cols_ns]).std()
    t = pd.DataFrame({
        "n_bins": [len(sc_e.bins_[c]["cuts"]) + 1 for c in cols_ns],
        "subscore_sd": sd.values,
        "w_equal": sc_e.weights_.values,
        "w_learned": sc_l.weights_.values,
    }, index=cols_ns)
    t["influence_equal_%"] = 100 * t["w_equal"] * t["subscore_sd"] / (t["w_equal"] * t["subscore_sd"]).sum()
    t["influence_learned_%"] = 100 * t["w_learned"] * t["subscore_sd"] / (t["w_learned"] * t["subscore_sd"]).sum()
    weight_tables[pts] = t.sort_values("w_learned", ascending=False)

wt = weight_tables["equal"]
print("Equal-spacing points + both weightings (sorted by learned weight, top 15)")
print(wt.head(15).round(3).to_string())
print(f"\nVariables with learned weight = 0: {(wt['w_learned'] == 0).sum()} / {len(wt)}")
print("\nNumber of bins vs. influence (average influence %):")
print(wt.groupby("n_bins")[["influence_equal_%", "influence_learned_%"]].mean().round(2).to_string())
wt.round(3)

Equal-spacing points + both weightings (sorted by learned weight, top 15)
                             n_bins  subscore_sd  w_equal  w_learned  influence_equal_%  influence_learned_%
under_level_approvals             3        1.415    0.167      1.969              0.808               11.775
system_sponsor_share              3        1.379    0.167      0.658              0.788                3.835
expert_hcp_consultant             2        2.516    0.167      0.524              1.437                5.569
n_business_units                  3        3.316    0.167      0.404              1.894                5.659
n_engagements                     5        3.323    0.167      0.391              1.898                5.482
expert_other                      2        1.530    0.167      0.388              0.874                2.509
activity_density                  5        3.541    0.167      0.363              2.022                5.437
sensitive_expense_share           3        3.028    0.

,n_bins,subscore_sd,w_equal,w_learned,influence_equal_%,influence_learned_%
under_level_approvals,3,1.415,0.167,1.969,0.808,11.775
system_sponsor_share,3,1.379,0.167,0.658,0.788,3.835
expert_hcp_consultant,2,2.516,0.167,0.524,1.437,5.569
n_business_units,3,3.316,0.167,0.404,1.894,5.659
n_engagements,5,3.323,0.167,0.391,1.898,5.482
expert_other,2,1.530,0.167,0.388,0.874,2.509
activity_density,5,3.541,0.167,0.363,2.022,5.437
sensitive_expense_share,3,3.028,0.167,0.360,1.730,4.599
high_band_share,4,4.201,0.167,0.356,2.399,6.316
expense_reimbursement_share,3,2.134,0.167,0.327,1.219,2.950


## Cell 22 · How the Weights Were Obtained

### 1. Why we can't just give every variable the same weight

How much a variable affects the **ranking** doesn't depend on its maximum (which is 10 for everyone). It depends on **how spread out vendors' scores are on that variable**, i.e. the standard deviation of its subscore. So each variable's actual influence is roughly

$$
\text{Influence}_j \;\propto\; w_j \times \text{sd}(\text{Subscore}_j)
$$

That standard deviation depends on two things: **how many bins** the variable has, and **how evenly vendors are spread across those bins**. For example, a 5-bin variable cut at quintiles has 20% of vendors in each bin, so scores are spread out a lot. A "concentrated" variable might have 3 bins, but with 80%–96% of vendors in the lowest bin, the scores barely spread at all. **With equal weights, a variable's influence is determined entirely by these two things, and has nothing to do with how important the variable actually is.**

Actual results from Cell 21 (equal-spacing points), averaged by number of bins:

| Bins | Avg. influence, equal weights | Avg. influence, learned weights |
|---|---|---|
| 2 bins | 1.81% | 1.32% |
| 3 bins | 1.15% | 2.14% |
| 4 bins | 2.38% | 3.40% |
| 5 bins | 2.03% | 1.40% |

(Most 3-bin variables are the "concentrated" type, which is why they actually have the *lowest* influence under equal weights.)

Two examples:

- `under_level_approvals` (approvals signed below the required level) has 3 bins, but 96% of vendors are in the lowest bin, so its subscore SD is only 1.42 and its influence under equal weights is just **0.8%**. Yet it's the strongest signal we have: all 38 vendors with a violation are positives.
- `activity_density` (payment activity) has 5 bins with vendors spread evenly, so its subscore SD is 3.54 and its influence under equal weights is **2.0%**, more than twice as much, even though it separates positives from negatives much less well.

So the weights **have to be adjusted**, and the adjustment shouldn't be based on "how many bins" but on "how much this variable actually helps separate positives from negatives".

### 2. Two kinds of weights

**(a) Equal weights:** every variable gets $w_j = 10/60 \approx 0.167$. This represents a purely manual "all variables matter equally" setup, and we use it as a comparison.

**(b) Learned weights:** let the data decide how much each variable should count. We treat the 60 subscores as predictors and fit a **logistic regression** on $y$:

$$
\log \frac{P(y_i = 1)}{1 - P(y_i = 1)} \;=\; b_0 + \sum_j \beta_j \cdot \frac{\text{Subscore}_{ij}}{10}
$$

Settings:

| Setting | Choice | Reason |
|---|---|---|
| Coefficient constraint | $\beta_j \ge 0$ | Negative weights don't make sense in a scorecard: a higher subscore should mean more suspicious. The direction was already set during binning, so the weight only decides "how important" |
| Regularization | L2, $C = 1$ | 60 variables and 159 positives; keeps the coefficients from getting too large or overfitting the practice set |
| Class weighting | Positives weighted ≈ 5.3× negatives | Same as T5 in Cell 6 |
| Rescaling | $w_j = 10 \cdot \beta_j / \sum_k \beta_k$ | Makes the weights sum to 10 so the total is 0–100; proportional rescaling doesn't change the ranking |

Learned weights **automatically correct for the influence differences caused by binning**: if a variable is finely binned but doesn't separate the classes well, its coefficient naturally shrinks, possibly to 0.

### 3. Learned weights (equal-spacing points, all data, top 8)

| Variable | Meaning | Bins | Equal weight | Learned weight | Influence (equal) | Influence (learned) |
|---|---|---|---|---|---|---|
| `under_level_approvals` | Approvals signed below the required level | 3 | 0.167 | **1.969** | 0.8% | 11.8% |
| `system_sponsor_share` | Share of engagements with the system-default sponsor | 3 | 0.167 | **0.658** | 0.8% | 3.8% |
| `expert_hcp_consultant` | Expert type: HCP consultant | 2 | 0.167 | **0.524** | 1.4% | 5.6% |
| `n_business_units` | Number of business units | 3 | 0.167 | **0.404** | 1.9% | 5.7% |
| `n_engagements` | Number of engagements | 5 | 0.167 | **0.391** | 1.9% | 5.5% |
| `expert_other` | Expert type: other (3 rare types merged) | 2 | 0.167 | **0.388** | 0.9% | 2.5% |
| `activity_density` | Months with payments / months since onboarding | 5 | 0.167 | **0.363** | 2.0% | 5.4% |
| `sensitive_expense_share` | Share of sensitive expenses (commissions/grants/sponsorships/gov. fees) | 3 | 0.167 | **0.360** | 1.7% | 4.6% |

- **14** of the 60 variables get a learned weight of 0: once the other variables are taken into account, they add nothing, so they effectively drop out of the scorecard.
- In cross-validation, the weights are re-learned on each training fold.

### 4. Four versions of Model 6

| Version | Points within a variable | Weights across variables | What it represents |
|---|---|---|---|
| 6a | Equal spacing | Equal | Purely manual rules |
| **6b (`model_ms`, main version)** | Equal spacing | Learned | Our original design: equally spaced bins, weights adjusted from data |
| 6c | By positive rate | Equal | |
| 6d | By positive rate | Learned | A standard data-driven scorecard |

Comparing 6a and 6b tells us whether the weights need adjusting; comparing 6b and 6d tells us whether equal spacing is reasonable.

In [11]:
# ============================================================
# Cell 23. model_ms (Model 6): can it pick the 159 positives out of 1,002 vendors?
# ============================================================
MS_VARIANTS = {
    "6a Scorecard equal-pts equal-wts": MultiLevelScorecard(points="equal", weights="equal"),
    "6b Scorecard equal-pts learned-wts": MultiLevelScorecard(points="equal", weights="learned"),
    "6c Scorecard rate-pts equal-wts": MultiLevelScorecard(points="rate", weights="equal"),
    "6d Scorecard rate-pts learned-wts": MultiLevelScorecard(points="rate", weights="learned"),
}
MS_NAME = "6b Scorecard equal-pts learned-wts"
model_ms = MS_VARIANTS[MS_NAME]

ms_summary = {}
for name, m in MS_VARIANTS.items():
    mean_, std_ = run(name, m, X)
    ms_summary[name] = mean_.loc["No-shortcut"]
ms_tbl = pd.DataFrame(ms_summary).T
print("No-shortcut, mean over 5 seeds")
print(ms_tbl[["AP", "P@10", "P@25", "P@50", "P@100", "R@100", "case_hit@100", "member_recall@100"]].to_string())

# ---- How many of the 159 positives are in the top k ----
def capture(score, ks=(50, 100, 159, 200, 300, 500)):
    order = score.sort_values(ascending=False).index
    return {k: int(y.loc[order[:k]].sum()) for k in ks}

cap = pd.DataFrame({
    "Random (expected)": {k: round(k * y.mean(), 1) for k in (50, 100, 159, 200, 300, 500)},
    "2 Logistic regression": capture(SCORES[("2 Logistic regression", "No-shortcut")]),
    "4 Random forest": capture(SCORES[("4 Random forest", "No-shortcut")]),
    **{n: capture(SCORES[(n, "No-shortcut")]) for n in MS_VARIANTS},
}).T
cap.columns = [f"top {k}" for k in cap.columns]
print(f"\nPositives found in the top k (out of {y.sum()}; No-shortcut; ranked by the 5-seed average score)")
print(cap.to_string())

# ---- Scorecard breakdown for one vendor (main version, fit on all data, for illustration) ----
ms_fit = clone(model_ms).fit(X[cols_ns], y)
total = ms_fit.total_score(X[cols_ns]).sort_values(ascending=False)
example = total.index[0]
detail = pd.DataFrame({"value": X.loc[example, cols_ns], "subscore": ms_fit.subscores(X[cols_ns]).loc[example],
                       "weight": ms_fit.weights_})
detail["points"] = detail["subscore"] * detail["weight"]
print(f"\nScorecard breakdown: {example}, total {total.iloc[0]:.1f} / 100 (y = {y[example]})")
print(detail.sort_values("points", ascending=False).head(8).round(2).to_string())
cap

No-shortcut, mean over 5 seeds
                                       AP  P@10   P@25   P@50  P@100  R@100  case_hit@100  member_recall@100
6a Scorecard equal-pts equal-wts    0.220  0.32  0.368  0.372  0.290  0.182         0.509              0.308
6b Scorecard equal-pts learned-wts  0.416  1.00  0.896  0.708  0.482  0.303         0.594              0.387
6c Scorecard rate-pts equal-wts     0.250  0.32  0.432  0.412  0.306  0.192         0.533              0.307
6d Scorecard rate-pts learned-wts   0.398  0.92  0.824  0.652  0.448  0.282         0.430              0.258

Positives found in the top k (out of 159; No-shortcut; ranked by the 5-seed average score)
                                    top 50  top 100  top 159  top 200  top 300  top 500
Random (expected)                      7.9     15.9     25.2     31.7     47.6     79.3
2 Logistic regression                 41.0     54.0     63.0     66.0     80.0    112.0
4 Random forest                       47.0     61.0     72.0     75.

,top 50,top 100,top 159,top 200,top 300,top 500
Random (expected),7.9,15.9,25.2,31.7,47.6,79.3
2 Logistic regression,41.0,54.0,63.0,66.0,80.0,112.0
4 Random forest,47.0,61.0,72.0,75.0,92.0,119.0
6a Scorecard equal-pts equal-wts,21.0,35.0,46.0,54.0,63.0,87.0
6b Scorecard equal-pts learned-wts,39.0,54.0,64.0,70.0,80.0,101.0
6c Scorecard rate-pts equal-wts,23.0,37.0,43.0,57.0,75.0,104.0
6d Scorecard rate-pts learned-wts,38.0,51.0,62.0,68.0,89.0,105.0


## Cell 24 · Comparing All Models (Model 1 Linear Regression → Model 6 Scorecard)

Four things we look at when reading the table:

1. **What each step adds:** 1 → 2 makes the output a probability; 2 → 3 allows thresholds and interactions; 3 → 4 averages many trees; 5 uses no labels; 6 is a fixed-structure scorecard.
2. **How far the scorecard is from the statistical models:** this is the gap between "traditional rule-based scoring" and "letting a model learn".
3. **Full vs. No-shortcut:** a bigger gap means the model relies more on variables that might be practice-set shortcuts.
4. **Standard deviations:** if the gap between two models is smaller than the SD, we can't say one is better than the other.

In [12]:
# ============================================================
# Cell 25. Comparison table
# ============================================================
allres = pd.concat(RESULTS, ignore_index=True)
metrics = ["AP", "P@10", "P@25", "P@50", "P@100", "R@100", "first_hit",
           "case_hit@100", "member_recall@100"]
comp_mean = allres.groupby(["features", "model"], sort=False)[metrics].mean().round(3)
comp_std = allres.groupby(["features", "model"], sort=False)[["AP", "P@50"]].std().round(3)
comp = comp_mean.join(comp_std, rsuffix="_std")
print(f"random precision = {y.mean():.3f}\n")
for fs in FEATURE_SETS:
    print(f"===== {fs} =====")
    print(comp.loc[fs].to_string()); print()
comp

random precision = 0.159

===== Full =====
                                       AP  P@10   P@25   P@50  P@100  R@100  first_hit  case_hit@100  member_recall@100  AP_std  P@50_std
model                                                                                                                                    
Random                              0.166  0.12  0.168  0.168  0.164  0.103        5.6         0.345              0.095   0.009     0.018
1 Linear regression                 0.589  1.00  0.992  0.796  0.638  0.401        1.0         0.679              0.420   0.025     0.048
2 Logistic regression               0.602  1.00  0.976  0.808  0.660  0.415        1.0         0.691              0.412   0.021     0.036
3 Decision tree                     0.419  0.72  0.592  0.600  0.562  0.353        1.0         0.558              0.374   0.011     0.040
4 Random forest                     0.578  0.76  0.704  0.704  0.684  0.430        1.2         0.691              0.484   0.014  

,,AP,P@10,P@25,P@50,P@100,R@100,first_hit,case_hit@100,member_recall@100,AP_std,P@50_std
features,model,,,,,,,,,,,
Full,Random,0.166,0.12,0.168,0.168,0.164,0.103,5.6,0.345,0.095,0.009,0.018
No-shortcut,Random,0.166,0.12,0.168,0.168,0.164,0.103,5.6,0.345,0.095,0.009,0.018
Full,1 Linear regression,0.589,1.00,0.992,0.796,0.638,0.401,1.0,0.679,0.420,0.025,0.048
No-shortcut,1 Linear regression,0.487,1.00,1.000,0.892,0.556,0.350,1.0,0.527,0.386,0.020,0.030
Full,2 Logistic regression,0.602,1.00,0.976,0.808,0.660,0.415,1.0,0.691,0.412,0.021,0.036
No-shortcut,2 Logistic regression,0.462,1.00,1.000,0.812,0.504,0.317,1.0,0.533,0.362,0.020,0.036
Full,3 Decision tree,0.419,0.72,0.592,0.600,0.562,0.353,1.0,0.558,0.374,0.011,0.040
No-shortcut,3 Decision tree,0.467,0.96,0.968,0.900,0.558,0.351,1.0,0.485,0.315,0.008,0.037
Full,4 Random forest,0.578,0.76,0.704,0.704,0.684,0.430,1.2,0.691,0.484,0.014,0.052


## Cell 26 · Conclusions

All numbers are averages over 5 seeds; a random ranking gets AP ≈ 0.16. We mainly look at the **No-shortcut** results (without the 7 variables that might be practice-set shortcuts).

### 1. All models

| Model | AP (No-shortcut) | AP (Full) | Positives in the top 159 (out of 159) |
|---|---|---|---|
| Random ranking | 0.166 | 0.166 | ~25 |
| 1 Linear regression | 0.487 | 0.589 | – |
| 2 Logistic regression | 0.462 | 0.602 | 63 |
| 3 Decision tree | 0.467 | 0.419 | – |
| 4 Random forest | **0.510** | 0.578 | **72** |
| 5 Isolation forest (unsupervised) | 0.182 | 0.195 | – |
| 6a Scorecard: equal points + equal weights | 0.220 | 0.292 | 46 |
| **6b Scorecard: equal points + learned weights (`model_ms`)** | 0.416 | 0.509 | 64 |
| 6c Scorecard: rate points + equal weights | 0.250 | 0.336 | 43 |
| 6d Scorecard: rate points + learned weights | 0.398 | 0.559 | 62 |

### 2. What we learned about the scorecard (Model 6)

1. **The weights definitely need to be adjusted.** With equal weights (6a, 6c) the scorecard only reaches AP 0.22–0.25, barely better than random. With learned weights (6b, 6d) it goes up to 0.40–0.42, almost double. Cell 22 explains why: with equal weights, a variable's influence depends on how spread out its subscores are, not on how important it is. Our strongest signal, `under_level_approvals`, only gets 0.8% of the influence with equal weights, and about 12% once the weights are learned.
2. **Equal spacing within a variable seems acceptable.** On No-shortcut, equal spacing (6b, 0.416) and rate-based points (6d, 0.398) are within one standard deviation of each other (about 0.02–0.03). On Full, rate-based points do better (0.559 vs. 0.509). The main weakness of equal spacing is with variables where "going from 0 to 1" is the big jump: for example, a vendor with just one approval violation only gets half the points (Cell 20).
3. **At the top of the list, the scorecard does about as well as logistic regression.** In the top 159, 6b finds 64 positives and logistic regression finds 63, and both get the top 10 completely right. The overall AP is about 0.05 lower, which we think is the cost of turning continuous values into bins and losing the differences within a bin. What we get in return is **full transparency**: every vendor's total can be broken down into "19.7 points from approvals signed below the required level + the other items..." (see the breakdown in Cell 23), which an analyst can check directly.
4. **14 variables get a learned weight of 0.** Once the other variables are accounted for, they add nothing, so the scorecard effectively uses 46 variables.

### 3. What we learned about Layer 1 overall

1. **Looking at single rows finds some of the positives, but not all of them.** The best model (random forest) finds 72 of the 159 positives in its top 159, about 45%; the scorecard and logistic regression find about 40%; a random ranking finds only 16%. More than half of the positives look just like ordinary vendors when you only look at their own row.
2. **The bottleneck is information, not the model.** On No-shortcut, all the supervised models (Models 1–4, 6b, 6d) land between AP 0.40 and 0.51. Going from the scorecard to a random forest adds a lot of complexity but only gains about 0.1.
3. **The unsupervised model barely finds anything** (isolation forest AP 0.18). The members of the practice cases don't look "anomalous" on their own; the problem is in how they are connected.
4. **We can find case members, but not complete cases.** The supervised models only cover about 26%–39% of each case's members in their top 100. Connecting members into full cases needs the relational information from Layer 2.
5. **The shortcut variables matter a lot.** For most models, going from Full to No-shortcut drops AP by 0.07–0.16. We think the No-shortcut results should be treated as the real Layer 1 baseline.

### 4. Models we plan to carry into the next layers

In each later layer we want to **only add variables and keep the model settings the same**, so that any improvement can be attributed entirely to the new relational information. We plan to keep three models:

- **Logistic regression:** the interpretable baseline for the ladder comparison;
- **Random forest:** the best supervised model in Layer 1;
- **Multi-level scorecard 6b (`model_ms`):** represents traditional rule-based scoring, which lets us answer the main question of the project: can relational information find things that traditional scoring misses?


In [13]:
# ============================================================
# Cell 27. Export: Layer 1 baseline scores + reason codes
#
# Baseline model: Model 2 logistic regression on the No-shortcut variables
#   - later layers will be compared using "the same scoring model, only more variables",
#     so the baseline should be stable and interpretable;
#   - logistic regression coefficients split the score into per-variable contributions,
#     which we use for reason codes.
# Scores from all models (including the scorecard) are exported too, for comparison.
# ============================================================
BASE_MODEL, BASE_FS = "2 Logistic regression", "No-shortcut"
cols = FEATURE_SETS[BASE_FS]

Z = fit_logistic[:-1].transform(X_lin[cols])
names = list(cols) + [f"missing:{c}" for c in np.array(cols)[fit_logistic[0].indicator_.features_]]
contrib = pd.DataFrame(Z * fit_logistic[-1].coef_[0], index=X.index, columns=names)
desc = spec.set_index("variable")["meaning"]

def reasons(r, n=3):
    top = contrib.loc[r].sort_values(ascending=False).head(n)
    return "; ".join(f"{v} ({desc.get(v, v)}) = {X.at[r, v]:g}" if v in X.columns else v
                     for v in top.index if top[v] > 0)

ALL_MODELS = ["1 Linear regression", "2 Logistic regression", "3 Decision tree",
              "4 Random forest", "5 Isolation forest", *MS_VARIANTS]
out = pd.DataFrame({f"score_{m.split(' ', 1)[1].replace(' ', '_')}": SCORES[(m, BASE_FS)]
                    for m in ALL_MODELS})
out["risk_score"] = SCORES[(BASE_MODEL, BASE_FS)]
out["y"] = y
out = out.sort_values("risk_score", ascending=False)
out["rank"] = np.arange(1, len(out) + 1)
out["reason_codes"] = [reasons(r) for r in out.index]

submission = pd.DataFrame({
    "case_rank": out["rank"].to_numpy(),
    "member_record_ids": out.index,
    "typology_tags": "",                      # Layer 1 can't tell the typology
    "score": out["risk_score"].round(4).to_numpy(),
    "reason_codes": out["reason_codes"].to_numpy(),
})
Path("outputs").mkdir(exist_ok=True)
out.to_csv("outputs/layer1_scores.csv")
submission.to_csv("outputs/layer1_submission_format.csv", index=False)
X.assign(y=y).to_csv("outputs/layer1_features.csv")
allres.to_csv("outputs/layer1_model_results_by_seed.csv", index=False)
struct.to_csv("outputs/layer1_scorecard_bins.csv")
print("saved to outputs/: layer1_scores.csv, layer1_submission_format.csv, layer1_features.csv, "
      "layer1_model_results_by_seed.csv, layer1_scorecard_bins.csv")
out.head(10)

saved to outputs/: layer1_scores.csv, layer1_submission_format.csv, layer1_features.csv, layer1_model_results_by_seed.csv, layer1_scorecard_bins.csv


,score_Linear_regression,score_Logistic_regression,score_Decision_tree,score_Random_forest,score_Isolation_forest,score_Scorecard_equal-pts_equal-wts,score_Scorecard_equal-pts_learned-wts,score_Scorecard_rate-pts_equal-wts,score_Scorecard_rate-pts_learned-wts,risk_score,y,rank,reason_codes
record_id,,,,,,,,,,,,,
TP-E62F69EE,1.263696,0.999301,1.0,0.629952,0.481154,0.624167,0.730174,0.606487,0.619975,0.999301,1,1,under_level_approvals (Approvals signed below the required level) = 3; n_ins...
TP-FC036024,1.302219,0.999296,1.0,0.645533,0.422046,0.509167,0.736675,0.491958,0.613413,0.999296,1,2,under_level_approvals (Approvals signed below the required level) = 2; n_bus...
TP-ABC566F2,1.225184,0.998908,1.0,0.705864,0.407523,0.553889,0.663207,0.515853,0.571824,0.998908,1,3,under_level_approvals (Approvals signed below the required level) = 2; activ...
TP-B1885BF8,1.232356,0.998522,1.0,0.664420,0.435236,0.543333,0.755770,0.490476,0.586377,0.998522,1,4,under_level_approvals (Approvals signed below the required level) = 7; sensi...
TP-D6CA8844,1.214077,0.998420,1.0,0.694679,0.430969,0.608611,0.764098,0.573200,0.609161,0.998420,1,5,under_level_approvals (Approvals signed below the required level) = 3; sensi...
TP-E4DEA4DF,1.239347,0.998408,1.0,0.769202,0.488869,0.608611,0.650686,0.618616,0.650896,0.998408,1,6,under_level_approvals (Approvals signed below the required level) = 1; n_ins...
TP-907B6E23,1.170022,0.998357,1.0,0.697107,0.439321,0.575278,0.746001,0.579011,0.637013,0.998357,1,7,under_level_approvals (Approvals signed below the required level) = 3; sensi...
TP-2FCE3667,1.143218,0.997729,1.0,0.703096,0.493230,0.615833,0.647159,0.609775,0.621214,0.997729,1,8,under_level_approvals (Approvals signed below the required level) = 1; n_ins...
TP-67809112,1.111589,0.996146,1.0,0.668741,0.440057,0.557778,0.727168,0.556706,0.629651,0.996146,1,9,under_level_approvals (Approvals signed below the required level) = 3; sensi...
